In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, ndimage
from scipy.fft import fft, fft2, fftshift, ifft2
from scipy.io import wavfile

# =============================================================================
# PART 1: AUDIO PROCESSING
# =============================================================================

# Generate synthetic audio signals
fs = 44100  # CD quality sampling rate
duration = 3.0
t = np.linspace(0, duration, int(fs * duration), endpoint=False)

# 1. Musical note synthesis (A4 = 440 Hz with harmonics)
fundamental = 440
audio_note = (np.sin(2*np.pi*fundamental*t) +          # Fundamental
              0.5*np.sin(2*np.pi*2*fundamental*t) +    # 2nd harmonic
              0.25*np.sin(2*np.pi*3*fundamental*t))    # 3rd harmonic

# 2. Speech-like signal (formants)
# Simulate vowel sound with multiple formants
formant1 = signal.chirp(t, 700, duration, 720, method='linear')
formant2 = signal.chirp(t, 1220, duration, 1250, method='linear')
formant3 = signal.chirp(t, 2600, duration, 2650, method='linear')
audio_speech = 0.4*formant1 + 0.3*formant2 + 0.2*formant3

# 3. Add background noise
noise = 0.05 * np.random.randn(len(t))
audio_noisy = audio_note + noise

# Audio processing techniques
# A. Spectral subtraction for noise reduction
window_size = 2048
hop_size = window_size // 2

f_noise, t_noise, Sxx_noisy = signal.spectrogram(audio_noisy, fs, 
                                                  nperseg=window_size, 
                                                  noverlap=window_size-hop_size)
f_clean, t_clean, Sxx_clean = signal.spectrogram(audio_note, fs, 
                                                  nperseg=window_size, 
                                                  noverlap=window_size-hop_size)

# B. Pitch detection using autocorrelation
def detect_pitch(signal, fs, fmin=80, fmax=800):
    """Detect pitch using autocorrelation method"""
    # Autocorrelation
    corr = np.correlate(signal, signal, mode='full')
    corr = corr[len(corr)//2:]
    
    # Find peaks in autocorrelation
    d_min = int(fs / fmax)
    d_max = int(fs / fmin)
    
    # Find maximum in the valid range
    peak_idx = np.argmax(corr[d_min:d_max]) + d_min
    pitch = fs / peak_idx
    return pitch

pitch_detected = detect_pitch(audio_note[:fs], fs)

# C. Audio effects - Reverb (using convolution)
# Simple reverb using exponentially decaying impulse response
reverb_length = int(0.5 * fs)  # 0.5 second reverb
reverb_ir = np.exp(-5 * np.linspace(0, 1, reverb_length)) * np.random.randn(reverb_length) * 0.3
audio_reverb = signal.fftconvolve(audio_note[:fs], reverb_ir, mode='same')

# =============================================================================
# PART 2: IMAGE PROCESSING
# =============================================================================

# Create synthetic images
size = 256

# 1. Geometric patterns
x = np.linspace(-5, 5, size)
y = np.linspace(-5, 5, size)
X, Y = np.meshgrid(x, y)
image1 = np.sin(np.sqrt(X**2 + Y**2)) * np.exp(-0.1*(X**2 + Y**2))

# 2. Add noise
image_noisy = image1 + 0.2 * np.random.randn(size, size)

# Image processing techniques
# A. 2D Fourier Transform for frequency analysis
image_fft = fft2(image_noisy)
image_fft_shifted = fftshift(image_fft)
magnitude_spectrum = np.log(np.abs(image_fft_shifted) + 1)

# B. Filtering in frequency domain
# Low-pass filter (removes high frequencies = smoothing)
rows, cols = image_noisy.shape
crow, ccol = rows//2, cols//2
radius = 30
mask_lowpass = np.zeros((rows, cols))
y, x = np.ogrid[:rows, :cols]
mask_area = (x - ccol)**2 + (y - crow)**2 <= radius**2
mask_lowpass[mask_area] = 1

# Apply low-pass filter
fft_filtered_low = image_fft_shifted * mask_lowpass
image_filtered_low = np.real(ifft2(fftshift(fft_filtered_low)))

# High-pass filter (edge enhancement)
mask_highpass = 1 - mask_lowpass
fft_filtered_high = image_fft_shifted * mask_highpass
image_filtered_high = np.real(ifft2(fftshift(fft_filtered_high)))

# C. Edge detection using various methods
# Sobel edge detection
edges_sobel_x = ndimage.sobel(image1, axis=0)
edges_sobel_y = ndimage.sobel(image1, axis=1)
edges_sobel = np.hypot(edges_sobel_x, edges_sobel_y)

# Laplacian edge detection
edges_laplacian = ndimage.laplace(image1)

# Canny edge detection (using Gaussian derivative)
edges_canny = ndimage.gaussian_filter(image1, sigma=2)
edges_canny = ndimage.gaussian_gradient_magnitude(edges_canny, sigma=1)

# D. Image enhancement
# Unsharp masking
blurred = ndimage.gaussian_filter(image1, sigma=2)
sharpened = image1 + 1.5 * (image1 - blurred)

# =============================================================================
# VISUALIZATION
# =============================================================================

fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(4, 4, hspace=0.3, wspace=0.3)

# Row 1: Audio signals
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t[:fs//10], audio_note[:fs//10], linewidth=1)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.set_title(f'Musical Note A4 ({fundamental} Hz)\nwith Harmonics')
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(t[:fs//10], audio_speech[:fs//10], linewidth=1, color='green')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Amplitude')
ax2.set_title('Synthetic Speech-like Signal\n(Formant Structure)')
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(t[:fs//10], audio_noisy[:fs//10], linewidth=0.5, alpha=0.7, color='red')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Amplitude')
ax3.set_title('Noisy Audio Signal')
ax3.grid(True, alpha=0.3)

ax4 = fig.add_subplot(gs[0, 3])
ax4.plot(t[:fs//10], audio_reverb[:fs//10], linewidth=1, color='purple')
ax4.set_xlabel('Time (s)')
ax4.set_ylabel('Amplitude')
ax4.set_title('Audio with Reverb Effect')
ax4.grid(True, alpha=0.3)

# Row 2: Audio spectrograms
ax5 = fig.add_subplot(gs[1, :2])
im1 = ax5.pcolormesh(t_noisy, f_noise, 10*np.log10(Sxx_noisy + 1e-10), 
                      shading='gouraud', cmap='viridis')
ax5.set_ylabel('Frequency (Hz)')
ax5.set_xlabel('Time (s)')
ax5.set_title('Spectrogram: Noisy Audio')
ax5.set_ylim([0, 2000])
plt.colorbar(im1, ax=ax5, label='Power (dB)')

ax6 = fig.add_subplot(gs[1, 2:])
# Speech spectrogram
f_sp, t_sp, Sxx_sp = signal.spectrogram(audio_speech, fs, nperseg=window_size, 
                                         noverlap=window_size-hop_size)
im2 = ax6.pcolormesh(t_sp, f_sp, 10*np.log10(Sxx_sp + 1e-10), 
                      shading='gouraud', cmap='jet')
ax6.set_ylabel('Frequency (Hz)')
ax6.set_xlabel('Time (s)')
ax6.set_title('Spectrogram: Speech-like Signal (Formants visible)')
ax6.set_ylim([0, 4000])
plt.colorbar(im2, ax=ax6, label='Power (dB)')

# Row 3: Image processing - Frequency domain
ax7 = fig.add_subplot(gs[2, 0])
ax7.imshow(image_noisy, cmap='gray')
ax7.set_title('Original Noisy Image')
ax7.axis('off')

ax8 = fig.add_subplot(gs[2, 1])
ax8.imshow(magnitude_spectrum, cmap='hot')
ax8.set_title('2D FFT Magnitude Spectrum\n(Frequency Domain)')
ax8.axis('off')

ax9 = fig.add_subplot(gs[2, 2])
ax9.imshow(image_filtered_low, cmap='gray')
ax9.set_title('Low-pass Filtered\n(Smoothed)')
ax9.axis('off')

ax10 = fig.add_subplot(gs[2, 3])
ax10.imshow(np.abs(image_filtered_high), cmap='gray')
ax10.set_title('High-pass Filtered\n(Edges Enhanced)')
ax10.axis('off')

# Row 4: Edge detection and enhancement
ax11 = fig.add_subplot(gs[3, 0])
ax11.imshow(edges_sobel, cmap='gray')
ax11.set_title('Sobel Edge Detection')
ax11.axis('off')

ax12 = fig.add_subplot(gs[3, 1])
ax12.imshow(edges_laplacian, cmap='gray')
ax12.set_title('Laplacian Edge Detection')
ax12.axis('off')

ax13 = fig.add_subplot(gs[3, 2])
ax13.imshow(edges_canny, cmap='gray')
ax13.set_title('Gradient Magnitude\n(Canny-like)')
ax13.axis('off')

ax14 = fig.add_subplot(gs[3, 3])
ax14.imshow(sharpened, cmap='gray')
ax14.set_title('Unsharp Masking\n(Enhanced)')
ax14.axis('off')

plt.savefig('audio_image_processing_applications.png', dpi=150, bbox_inches='tight')
plt.show()

# Print comprehensive summary
print("="*70)
print("AUDIO AND IMAGE PROCESSING WITH SIGNAL PROCESSING")
print("="*70)

print("\n" + "="*70)
print("AUDIO PROCESSING")
print("="*70)

print(f"\n1. AUDIO PARAMETERS:")
print(f"   Sampling Rate: {fs} Hz (CD quality)")
print(f"   Duration: {duration} seconds")
print(f"   Total Samples: {len(t):,}")
print(f"   Fundamental Frequency: {fundamental} Hz (A4 note)")
print(f"   Detected Pitch: {pitch_detected:.2f} Hz")

print(f"\n2. AUDIO PROCESSING TECHNIQUES DEMONSTRATED:")
print(f"   - Harmonic synthesis (additive synthesis)")
print(f"   - Formant modeling (speech simulation)")
print(f"   - Spectral analysis (STFT/Spectrogram)")
print(f"   - Pitch detection (autocorrelation)")
print(f"   - Audio effects (reverb via convolution)")
print(f"   - Noise reduction (spectral subtraction concept)")

print(f"\n3. SPECTROGRAM PARAMETERS:")
print(f"   Window size: {window_size} samples ({window_size/fs*1000:.2f} ms)")
print(f"   Hop size: {hop_size} samples")
print(f"   Frequency resolution: {fs/window_size:.2f} Hz")
print(f"   Time resolution: {hop_size/fs*1000:.2f} ms")

print("\n" + "="*70)
print("IMAGE PROCESSING")
print("="*70)

print(f"\n4. IMAGE PARAMETERS:")
print(f"   Image Size: {size}×{size} pixels")
print(f"   Total Pixels: {size*size:,}")

print(f"\n5. IMAGE PROCESSING TECHNIQUES DEMONSTRATED:")
print(f"   - 2D Fourier Transform (frequency analysis)")
print(f"   - Frequency domain filtering (low-pass, high-pass)")
print(f"   - Edge detection (Sobel, Laplacian, Gradient)")
print(f"   - Image enhancement (unsharp masking)")
print(f"   - Noise reduction (smoothing)")

print(f"\n6. FREQUENCY DOMAIN FILTERING:")
print(f"   Low-pass radius: {radius} pixels")
print(f"   Effect: Removes high frequencies → Image smoothing")
print(f"   High-pass: Removes low frequencies → Edge enhancement")

print("\n" + "="*70)
print("CONNECTIONS TO DEEP LEARNING")
print("="*70)

print("\n7. AUDIO ML APPLICATIONS:")
print("   - Speech Recognition: MFCC + RNN/Transformer")
print("   - Music Generation: WaveNet, MusicVAE")
print("   - Audio Classification: Spectrogram + CNN")
print("   - Voice Conversion: CycleGAN with spectral features")
print("   - Sound Event Detection: Mel-spectrograms + ResNet")

print("\n8. IMAGE ML APPLICATIONS:")
print("   - Object Detection: CNN with learned convolutional filters")
print("   - Image Segmentation: U-Net (encoder-decoder with skip connections)")
print("   - Style Transfer: VGG features (learned frequency representations)")
print("   - Super-resolution: Frequency domain constraints")
print("   - Image Denoising: Autoencoders + wavelet decomposition")

print("\n9. KEY INSIGHT:")
print("   Traditional signal processing ≡ Hand-crafted features")
print("   Deep Learning ≡ Learned features (but still based on convolution!)")
print("   Modern AI: Combines both approaches for best results")

### 5.8 Audio and Image Processing Applications

Signal processing is the foundation of modern audio and image processing in AI systems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, fft
from scipy.fft import fft2, ifft2, fftshift
import pywt

# =============================================================================
# 1. Time-Domain Features (commonly used in ML)
# =============================================================================

# Generate example signals
fs = 1000
t = np.linspace(0, 2, 2*fs, endpoint=False)

# Different signal classes for classification
signal_1 = np.sin(2*np.pi*10*t) + 0.5*np.sin(2*np.pi*30*t)  # Class 1: Low freq
signal_2 = np.sin(2*np.pi*100*t) + 0.5*np.sin(2*np.pi*150*t)  # Class 2: High freq
signal_3 = signal.chirp(t, 10, 2, 200)  # Class 3: Chirp

def extract_time_features(x):
    """Extract time-domain features for ML"""
    features = {}
    features['mean'] = np.mean(x)
    features['std'] = np.std(x)
    features['var'] = np.var(x)
    features['rms'] = np.sqrt(np.mean(x**2))
    features['peak'] = np.max(np.abs(x))
    features['peak_to_peak'] = np.ptp(x)
    features['crest_factor'] = np.max(np.abs(x)) / np.sqrt(np.mean(x**2))
    features['skewness'] = np.mean(((x - np.mean(x)) / np.std(x))**3)
    features['kurtosis'] = np.mean(((x - np.mean(x)) / np.std(x))**4)
    features['energy'] = np.sum(x**2)
    
    # Zero crossing rate
    features['zcr'] = np.sum(np.diff(np.sign(x)) != 0) / len(x)
    
    return features

# =============================================================================
# 2. Frequency-Domain Features
# =============================================================================

def extract_frequency_features(x, fs):
    """Extract frequency-domain features for ML"""
    features = {}
    
    # Compute FFT
    X = fft.fft(x)
    freqs = fft.fftfreq(len(x), 1/fs)
    magnitude = np.abs(X[:len(X)//2])
    freqs_pos = freqs[:len(freqs)//2]
    
    # Spectral features
    features['spectral_centroid'] = np.sum(freqs_pos * magnitude) / np.sum(magnitude)
    features['spectral_spread'] = np.sqrt(np.sum(((freqs_pos - features['spectral_centroid'])**2) * magnitude) / np.sum(magnitude))
    features['spectral_rolloff'] = freqs_pos[np.where(np.cumsum(magnitude) >= 0.85 * np.sum(magnitude))[0][0]]
    features['spectral_flux'] = np.sum(np.diff(magnitude)**2)
    
    # Dominant frequency
    features['dominant_freq'] = freqs_pos[np.argmax(magnitude)]
    features['spectral_flatness'] = np.exp(np.mean(np.log(magnitude + 1e-10))) / (np.mean(magnitude) + 1e-10)
    
    return features

# =============================================================================
# 3. Wavelet Features
# =============================================================================

def extract_wavelet_features(x):
    """Extract wavelet-domain features for ML"""
    features = {}
    
    # Multi-level wavelet decomposition
    coeffs = pywt.wavedec(x, 'db4', level=4)
    
    # Energy in each subband
    for i, c in enumerate(coeffs):
        features[f'wavelet_energy_level_{i}'] = np.sum(c**2)
        features[f'wavelet_std_level_{i}'] = np.std(c)
    
    # Total wavelet energy
    features['total_wavelet_energy'] = sum([np.sum(c**2) for c in coeffs])
    
    return features

# =============================================================================
# 4. Demonstrate CNN-like Convolution Operations
# =============================================================================

# Create simple 1D convolutional filters (like CNN kernels)
# These are analogous to learned filters in CNNs
filters_1d = {
    'Low-pass (Averaging)': np.ones(5) / 5,
    'High-pass (Derivative)': np.array([-1, 0, 1]),
    'Edge Detection': np.array([1, 0, -1]),
    'Gaussian Smoothing': signal.windows.gaussian(9, 1.5),
}

# Create comprehensive visualization
fig = plt.figure(figsize=(18, 16))
gs = fig.add_gridspec(5, 3, hspace=0.4, wspace=0.3)

# Row 1: Signal classes and feature extraction
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(t[:500], signal_1[:500], label='Class 1: Low Freq', linewidth=2, alpha=0.7)
ax1.plot(t[:500], signal_2[:500], label='Class 2: High Freq', linewidth=2, alpha=0.7)
ax1.plot(t[:500], signal_3[:500], label='Class 3: Chirp', linewidth=2, alpha=0.7)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.set_title('Example Signal Classes for ML Classification')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Row 2: Feature comparison
features_1 = {**extract_time_features(signal_1), **extract_frequency_features(signal_1, fs)}
features_2 = {**extract_time_features(signal_2), **extract_frequency_features(signal_2, fs)}
features_3 = {**extract_time_features(signal_3), **extract_frequency_features(signal_3, fs)}

# Select key features for visualization
key_features = ['std', 'rms', 'zcr', 'spectral_centroid', 'dominant_freq', 'spectral_spread']

ax2 = fig.add_subplot(gs[1, 0])
x_pos = np.arange(len(key_features))
width = 0.25
ax2.bar(x_pos - width, [features_1[f] for f in key_features], width, label='Class 1', alpha=0.8)
ax2.bar(x_pos, [features_2[f] for f in key_features], width, label='Class 2', alpha=0.8)
ax2.bar(x_pos + width, [features_3[f] for f in key_features], width, label='Class 3', alpha=0.8)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(key_features, rotation=45, ha='right')
ax2.set_ylabel('Feature Value (normalized)')
ax2.set_title('Extracted Features Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Spectral features visualization
ax3 = fig.add_subplot(gs[1, 1:])
for sig, label in [(signal_1, 'Class 1'), (signal_2, 'Class 2'), (signal_3, 'Class 3')]:
    X = np.abs(fft.fft(sig))
    freqs = fft.fftfreq(len(sig), 1/fs)
    ax3.semilogy(freqs[:len(freqs)//2], X[:len(freqs)//2], linewidth=2, label=label, alpha=0.7)
ax3.set_xlabel('Frequency (Hz)')
ax3.set_ylabel('Magnitude (log scale)')
ax3.set_title('Frequency Domain Representation of Signal Classes')
ax3.set_xlim([0, 300])
ax3.legend()
ax3.grid(True, alpha=0.3)

# Row 3: CNN-like 1D Convolutions
test_signal = signal_1[:500]
ax4 = fig.add_subplot(gs[2, 0])
ax4.plot(test_signal, linewidth=2, color='black', alpha=0.7)
ax4.set_xlabel('Sample')
ax4.set_ylabel('Amplitude')
ax4.set_title('Input Signal (like CNN input)')
ax4.grid(True, alpha=0.3)

# Apply different filters (like CNN kernels)
filter_outputs = {}
for name, kernel in filters_1d.items():
    filtered = np.convolve(test_signal, kernel, mode='same')
    filter_outputs[name] = filtered

ax5 = fig.add_subplot(gs[2, 1])
for name, filtered in list(filter_outputs.items())[:2]:
    ax5.plot(filtered, linewidth=2, label=name, alpha=0.7)
ax5.set_xlabel('Sample')
ax5.set_ylabel('Amplitude')
ax5.set_title('CNN-like Filter Outputs (Feature Maps)')
ax5.legend()
ax5.grid(True, alpha=0.3)

ax6 = fig.add_subplot(gs[2, 2])
for name, filtered in list(filter_outputs.items())[2:]:
    ax6.plot(filtered, linewidth=2, label=name, alpha=0.7)
ax6.set_xlabel('Sample')
ax6.set_ylabel('Amplitude')
ax6.set_title('More CNN-like Filter Outputs')
ax6.legend()
ax6.grid(True, alpha=0.3)

# Row 4: 2D Convolution (Image Processing - CNN foundation)
# Create a simple 2D signal (image-like)
x_2d = np.linspace(-5, 5, 100)
y_2d = np.linspace(-5, 5, 100)
X_2d, Y_2d = np.meshgrid(x_2d, y_2d)
image = np.sin(np.sqrt(X_2d**2 + Y_2d**2)) + 0.2*np.random.randn(100, 100)

# Define 2D convolutional kernels (CNN-style)
# Sobel filters for edge detection
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])
gaussian_kernel = np.outer(signal.windows.gaussian(5, 1), signal.windows.gaussian(5, 1))

# Apply convolutions
edges_x = signal.convolve2d(image, sobel_x, mode='same', boundary='wrap')
edges_y = signal.convolve2d(image, sobel_y, mode='same', boundary='wrap')
smoothed = signal.convolve2d(image, gaussian_kernel / np.sum(gaussian_kernel), mode='same', boundary='wrap')

ax7 = fig.add_subplot(gs[3, 0])
im1 = ax7.imshow(image, cmap='gray', aspect='auto')
ax7.set_title('Input "Image" (2D Signal)')
plt.colorbar(im1, ax=ax7)

ax8 = fig.add_subplot(gs[3, 1])
im2 = ax8.imshow(edges_x, cmap='RdBu', aspect='auto')
ax8.set_title('Horizontal Edge Detection\n(Sobel X - like CNN filter)')
plt.colorbar(im2, ax=ax8)

ax9 = fig.add_subplot(gs[3, 2])
im3 = ax9.imshow(smoothed, cmap='gray', aspect='auto')
ax9.set_title('Gaussian Smoothing\n(like CNN pooling preparation)')
plt.colorbar(im3, ax=ax9)

# Row 5: Feature map hierarchy (mimicking CNN layers)
# Simulate multi-level feature extraction
ax10 = fig.add_subplot(gs[4, 0])
# Layer 1: Raw signal
ax10.plot(test_signal, linewidth=1, color='blue', alpha=0.7, label='Layer 0: Input')
# Layer 1: After convolution
conv1 = signal.convolve(test_signal, filters_1d['Gaussian Smoothing'], mode='same')
ax10.plot(conv1, linewidth=1, color='green', alpha=0.7, label='Layer 1: Conv')
ax10.set_xlabel('Sample')
ax10.set_ylabel('Amplitude')
ax10.set_title('Feature Hierarchy (like CNN layers)')
ax10.legend()
ax10.grid(True, alpha=0.3)

# MFCC-like features (commonly used in audio ML)
ax11 = fig.add_subplot(gs[4, 1:])
# Compute spectrogram
f_spec, t_spec, Sxx = signal.spectrogram(signal_1, fs, nperseg=256, noverlap=128)
# Apply mel-scale-like transformation (log scale)
mel_spec = np.log10(Sxx + 1e-10)
im4 = ax11.pcolormesh(t_spec, f_spec, mel_spec, shading='gouraud', cmap='viridis')
ax11.set_ylabel('Frequency (Hz)')
ax11.set_xlabel('Time (s)')
ax11.set_title('Spectrogram Features (MFCC-like, used in audio ML)')
ax11.set_ylim([0, 200])
plt.colorbar(im4, ax=ax11, label='Log Power')

plt.savefig('ml_applications_signal_processing.png', dpi=150, bbox_inches='tight')
plt.show()

# Print comprehensive feature summary
print("="*60)
print("SIGNAL PROCESSING FOR MACHINE LEARNING")
print("="*60)

print("\n1. TIME-DOMAIN FEATURES (for classification):")
for feat in ['mean', 'std', 'rms', 'zcr', 'crest_factor', 'energy']:
    print(f"   {feat:20s}: Class1={features_1[feat]:8.4f}, Class2={features_2[feat]:8.4f}, Class3={features_3[feat]:8.4f}")

print("\n2. FREQUENCY-DOMAIN FEATURES:")
for feat in ['spectral_centroid', 'dominant_freq', 'spectral_spread', 'spectral_rolloff']:
    print(f"   {feat:20s}: Class1={features_1[feat]:8.2f}, Class2={features_2[feat]:8.2f}, Class3={features_3[feat]:8.2f}")

print("\n3. CNN-SIGNAL PROCESSING CONNECTIONS:")
print("   - Convolution in CNNs ≡ Filtering in signal processing")
print("   - CNN kernels ≡ Filter coefficients")
print("   - Feature maps ≡ Filtered signals")
print("   - Pooling ≡ Downsampling/decimation")
print("   - Multiple channels ≡ Filter banks")

print("\n4. COMMON ML APPLICATIONS:")
print("   - Audio classification: MFCC features + spectrograms")
print("   - Time series prediction: Wavelet features + statistical features")
print("   - Anomaly detection: Spectral features + envelope analysis")
print("   - Speech recognition: STFT + mel-scale filterbanks")
print("   - ECG/EEG analysis: Wavelet decomposition + band power features")

print("\n5. FEATURE ENGINEERING PIPELINE:")
print("   Raw Signal → Preprocessing (filtering) → Feature Extraction")
print("   → Feature Selection → ML Model → Prediction")

print(f"\n6. Total features extracted: {len(features_1)} per signal")

### 5.7 Machine Learning Applications: Feature Extraction and CNN Connections

Signal processing techniques are fundamental to feature extraction in ML, especially in CNNs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from matplotlib.patches import Circle

# Design different types of digital filters
fs = 1000  # Sampling frequency

# 1. Butterworth filter (maximally flat passband)
order_butter = 4
cutoff_butter = 100  # Hz
b_butter, a_butter = signal.butter(order_butter, cutoff_butter, fs=fs)

# 2. Chebyshev Type I filter (equiripple passband)
order_cheby1 = 4
ripple_cheby1 = 0.5  # dB
b_cheby1, a_cheby1 = signal.cheby1(order_cheby1, ripple_cheby1, cutoff_butter, fs=fs)

# 3. Chebyshev Type II filter (equiripple stopband)
order_cheby2 = 4
atten_cheby2 = 40  # dB
b_cheby2, a_cheby2 = signal.cheby2(order_cheby2, atten_cheby2, cutoff_butter, fs=fs)

# 4. Elliptic filter (equiripple in both bands)
order_ellip = 4
ripple_ellip = 0.5  # dB
atten_ellip = 40  # dB
b_ellip, a_ellip = signal.ellip(order_ellip, ripple_ellip, atten_ellip, cutoff_butter, fs=fs)

# 5. FIR filter using window method
numtaps = 51
b_fir = signal.firwin(numtaps, cutoff_butter, fs=fs, window='hamming')
a_fir = [1.0]  # FIR filters have a=1

# Create comprehensive visualization
fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(4, 3, hspace=0.4, wspace=0.3)

# Row 1: Pole-Zero plots
filters = {
    'Butterworth': (b_butter, a_butter),
    'Chebyshev I': (b_cheby1, a_cheby1),
    'Chebyshev II': (b_cheby2, a_cheby2),
    'Elliptic': (b_ellip, a_ellip)
}

for idx, (name, (b, a)) in enumerate(filters.items()):
    ax = fig.add_subplot(gs[0, idx % 3]) if idx < 3 else fig.add_subplot(gs[1, 0])
    
    # Compute zeros and poles
    z, p, k = signal.tf2zpk(b, a)
    
    # Plot unit circle
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), 'k--', linewidth=1, alpha=0.3)
    
    # Plot zeros and poles
    ax.scatter(np.real(z), np.imag(z), s=80, marker='o', 
               facecolors='none', edgecolors='b', linewidths=2, label='Zeros')
    ax.scatter(np.real(p), np.imag(p), s=80, marker='x', 
               color='r', linewidths=2, label='Poles')
    
    ax.set_xlabel('Real')
    ax.set_ylabel('Imaginary')
    ax.set_title(f'Pole-Zero Plot: {name}')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.axis('equal')
    ax.set_xlim([-1.5, 1.5])
    ax.set_ylim([-1.5, 1.5])
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)

# Row 1, col 3: FIR filter pole-zero plot
ax = fig.add_subplot(gs[0, 2])
z_fir = np.roots(b_fir)
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), 'k--', linewidth=1, alpha=0.3)
ax.scatter(np.real(z_fir), np.imag(z_fir), s=40, marker='o', 
           facecolors='none', edgecolors='b', linewidths=1, label='Zeros')
ax.scatter(0, 0, s=80, marker='x', color='r', linewidths=2, label='Poles (at origin)')
ax.set_xlabel('Real')
ax.set_ylabel('Imaginary')
ax.set_title(f'Pole-Zero Plot: FIR (Window)')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)
ax.axis('equal')
ax.set_xlim([-1.5, 1.5])
ax.set_ylim([-1.5, 1.5])
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)

# Row 2: Frequency responses (magnitude)
ax = fig.add_subplot(gs[1, 1:])
for name, (b, a) in filters.items():
    w, h = signal.freqz(b, a, worN=2048, fs=fs)
    ax.plot(w, 20*np.log10(np.abs(h)), linewidth=2, label=name, alpha=0.8)

w_fir, h_fir = signal.freqz(b_fir, a_fir, worN=2048, fs=fs)
ax.plot(w_fir, 20*np.log10(np.abs(h_fir)), linewidth=2, label='FIR', alpha=0.8)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('Frequency Response Comparison (Magnitude)')
ax.set_xlim([0, 250])
ax.set_ylim([-80, 5])
ax.axhline(-3, color='k', linestyle='--', alpha=0.5, linewidth=1)
ax.axvline(cutoff_butter, color='k', linestyle='--', alpha=0.5, linewidth=1)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Row 3: Phase responses
ax = fig.add_subplot(gs[2, 0])
for name, (b, a) in [('Butterworth', (b_butter, a_butter)), 
                      ('Chebyshev I', (b_cheby1, a_cheby1))]:
    w, h = signal.freqz(b, a, worN=2048, fs=fs)
    phase = np.unwrap(np.angle(h))
    ax.plot(w, phase, linewidth=2, label=name, alpha=0.8)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Phase (radians)')
ax.set_title('Phase Response (IIR Filters)')
ax.set_xlim([0, 250])
ax.legend()
ax.grid(True, alpha=0.3)

# Group delay
ax = fig.add_subplot(gs[2, 1])
for name, (b, a) in [('Butterworth', (b_butter, a_butter)), 
                      ('FIR', (b_fir, a_fir))]:
    w, gd = signal.group_delay((b, a), fs=fs)
    ax.plot(w, gd, linewidth=2, label=name, alpha=0.8)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Group Delay (samples)')
ax.set_title('Group Delay Comparison')
ax.set_xlim([0, 250])
ax.legend()
ax.grid(True, alpha=0.3)

# Impulse response
ax = fig.add_subplot(gs[2, 2])
for name, (b, a) in [('Butterworth', (b_butter, a_butter)), 
                      ('FIR', (b_fir, a_fir))]:
    _, y = signal.dimpulse((b, a, 1/fs), n=100)
    if isinstance(y, tuple):
        y = y[0]
    ax.plot(y.flatten(), linewidth=2, label=name, alpha=0.8)

ax.set_xlabel('Sample')
ax.set_ylabel('Amplitude')
ax.set_title('Impulse Response')
ax.legend()
ax.grid(True, alpha=0.3)

# Row 4: Filter application demonstration
# Create test signal with multiple frequencies
t = np.linspace(0, 2, 2*fs, endpoint=False)
test_signal = (np.sin(2*np.pi*50*t) +      # 50 Hz - in passband
               0.5*np.sin(2*np.pi*150*t) +  # 150 Hz - in transition
               0.3*np.sin(2*np.pi*300*t))   # 300 Hz - in stopband

# Apply different filters
y_butter = signal.filtfilt(b_butter, a_butter, test_signal)
y_cheby1 = signal.filtfilt(b_cheby1, a_cheby1, test_signal)
y_ellip = signal.filtfilt(b_ellip, a_ellip, test_signal)
y_fir = signal.filtfilt(b_fir, a_fir, test_signal)

ax = fig.add_subplot(gs[3, :2])
ax.plot(t[:500], test_signal[:500], 'k-', linewidth=1, alpha=0.3, label='Original')
ax.plot(t[:500], y_butter[:500], linewidth=1.5, label='Butterworth', alpha=0.8)
ax.plot(t[:500], y_ellip[:500], linewidth=1.5, label='Elliptic', alpha=0.8)
ax.plot(t[:500], y_fir[:500], linewidth=1.5, label='FIR', alpha=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Filtered Signal Comparison (Time Domain)')
ax.legend()
ax.grid(True, alpha=0.3)

# Frequency domain comparison
ax = fig.add_subplot(gs[3, 2])
from scipy.fft import fft, fftfreq
freqs = fftfreq(len(test_signal), 1/fs)
X_orig = np.abs(fft(test_signal))
X_butter = np.abs(fft(y_butter))
X_fir = np.abs(fft(y_fir))

ax.semilogy(freqs[:len(freqs)//2], X_orig[:len(freqs)//2], 
            'k-', linewidth=1, alpha=0.5, label='Original')
ax.semilogy(freqs[:len(freqs)//2], X_butter[:len(freqs)//2], 
            linewidth=2, label='Butterworth', alpha=0.8)
ax.semilogy(freqs[:len(freqs)//2], X_fir[:len(freqs)//2], 
            linewidth=2, label='FIR', alpha=0.8)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude (log scale)')
ax.set_title('Filtered Signal Spectrum')
ax.set_xlim([0, 400])
ax.axvline(50, color='g', linestyle='--', alpha=0.3, linewidth=1)
ax.axvline(150, color='orange', linestyle='--', alpha=0.3, linewidth=1)
ax.axvline(300, color='r', linestyle='--', alpha=0.3, linewidth=1)
ax.legend()
ax.grid(True, alpha=0.3)

plt.savefig('z_transform_digital_filters_comprehensive.png', dpi=150, bbox_inches='tight')
plt.show()

# Print filter characteristics
print("Digital Filter Design Summary:")
print(f"\n1. Butterworth Filter (Order {order_butter}):")
print(f"   - Maximally flat passband")
print(f"   - Monotonic response")
print(f"   - Poles: {len(signal.tf2zpk(b_butter, a_butter)[1])}")

print(f"\n2. Chebyshev Type I Filter (Order {order_cheby1}):")
print(f"   - Passband ripple: {ripple_cheby1} dB")
print(f"   - Sharper transition than Butterworth")
print(f"   - Poles: {len(signal.tf2zpk(b_cheby1, a_cheby1)[1])}")

print(f"\n3. Chebyshev Type II Filter (Order {order_cheby2}):")
print(f"   - Stopband attenuation: {atten_cheby2} dB")
print(f"   - Flat passband, ripple in stopband")
print(f"   - Poles: {len(signal.tf2zpk(b_cheby2, a_cheby2)[1])}")

print(f"\n4. Elliptic Filter (Order {order_ellip}):")
print(f"   - Passband ripple: {ripple_ellip} dB")
print(f"   - Stopband attenuation: {atten_ellip} dB")
print(f"   - Sharpest transition for given order")
print(f"   - Poles: {len(signal.tf2zpk(b_ellip, a_ellip)[1])}")

print(f"\n5. FIR Filter ({numtaps} taps):")
print(f"   - Linear phase (symmetric)")
print(f"   - Always stable")
print(f"   - Higher order needed for sharp transition")

print(f"\n6. Filter Design Parameters:")
print(f"   - Sampling frequency: {fs} Hz")
print(f"   - Cutoff frequency: {cutoff_butter} Hz")
print(f"   - Test signal components: 50, 150, 300 Hz")

### 5.6 Z-Transform and Digital Filter Design

The Z-transform is the discrete-time equivalent of the Laplace transform, essential for analyzing digital filters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, interpolate
from scipy.fft import fft, fftfreq

# Create a bandlimited continuous signal (in practice, densely sampled)
t_continuous = np.linspace(0, 1, 10000, endpoint=False)
f_signal = 50  # Signal frequency in Hz
x_continuous = np.sin(2*np.pi*f_signal*t_continuous) + 0.3*np.sin(2*np.pi*20*t_continuous)

# Nyquist frequency
f_nyquist = 2 * f_signal

# Sample at different rates
fs_under = 60      # Under-sampled (below Nyquist)
fs_nyquist = 100   # At Nyquist rate
fs_over = 500      # Over-sampled (above Nyquist)

# Generate sample points
n_under = int(fs_under * 1)
n_nyquist = int(fs_nyquist * 1)
n_over = int(fs_over * 1)

t_under = np.linspace(0, 1, n_under, endpoint=False)
t_nyquist = np.linspace(0, 1, n_nyquist, endpoint=False)
t_over = np.linspace(0, 1, n_over, endpoint=False)

x_under = np.sin(2*np.pi*f_signal*t_under) + 0.3*np.sin(2*np.pi*20*t_under)
x_nyquist = np.sin(2*np.pi*f_signal*t_nyquist) + 0.3*np.sin(2*np.pi*20*t_nyquist)
x_over = np.sin(2*np.pi*f_signal*t_over) + 0.3*np.sin(2*np.pi*20*t_over)

# Demonstrate aliasing
f_alias_test = 100  # Hz
fs_alias = 120      # Hz (below Nyquist for this frequency)
t_alias = np.linspace(0, 0.5, int(fs_alias * 0.5), endpoint=False)
x_alias_cont = np.sin(2*np.pi*f_alias_test*t_alias)
# The aliased frequency will appear at |f - fs| = |100 - 120| = 20 Hz
f_apparent = np.abs(f_alias_test - fs_alias)

# Signal reconstruction using sinc interpolation (ideal reconstruction)
def sinc_interp(x, s, t):
    """
    Sinc interpolation for ideal reconstruction
    x: sampled signal
    s: sample times
    t: times to reconstruct at
    """
    if len(x) != len(s):
        raise ValueError('x and s must have the same length')
    
    # Reshape for broadcasting
    T = np.tile(t, (len(s), 1))
    S = np.tile(s[:, np.newaxis], (1, len(t)))
    
    # Sinc interpolation formula
    return np.dot(x, np.sinc((T - S) * fs_over / np.pi))

# Reconstruct using sinc interpolation
t_recon = np.linspace(0, 1, 1000, endpoint=False)
x_recon_over = sinc_interp(x_over, t_over, t_recon)

# Create comprehensive visualization
fig = plt.figure(figsize=(16, 14))
gs = fig.add_gridspec(4, 3, hspace=0.4, wspace=0.3)

# Row 1: Different sampling rates
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t_continuous, x_continuous, 'b-', linewidth=1, alpha=0.5, label='Continuous')
ax1.stem(t_under, x_under, linefmt='r-', markerfmt='ro', basefmt='k-', label=f'Sampled at {fs_under} Hz')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.set_title(f'Under-sampling (fs={fs_under} Hz < Nyquist={f_nyquist} Hz)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 0.2])

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(t_continuous, x_continuous, 'b-', linewidth=1, alpha=0.5, label='Continuous')
ax2.stem(t_nyquist, x_nyquist, linefmt='g-', markerfmt='go', basefmt='k-', label=f'Sampled at {fs_nyquist} Hz')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Amplitude')
ax2.set_title(f'Nyquist Sampling (fs={fs_nyquist} Hz = Nyquist={f_nyquist} Hz)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 0.2])

ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(t_continuous, x_continuous, 'b-', linewidth=1, alpha=0.5, label='Continuous')
ax3.stem(t_over[:40], x_over[:40], linefmt='m-', markerfmt='mo', basefmt='k-', label=f'Sampled at {fs_over} Hz')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Amplitude')
ax3.set_title(f'Over-sampling (fs={fs_over} Hz > Nyquist={f_nyquist} Hz)')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_xlim([0, 0.2])

# Row 2: Frequency domain - aliasing demonstration
X_under = fft(x_under)
X_nyquist = fft(x_nyquist)
X_over = fft(x_over)

freqs_under = fftfreq(n_under, 1/fs_under)
freqs_nyquist = fftfreq(n_nyquist, 1/fs_nyquist)
freqs_over = fftfreq(n_over, 1/fs_over)

ax4 = fig.add_subplot(gs[1, 0])
ax4.stem(freqs_under[:n_under//2], np.abs(X_under[:n_under//2]), linefmt='r-', markerfmt='ro', basefmt='k-')
ax4.set_xlabel('Frequency (Hz)')
ax4.set_ylabel('Magnitude')
ax4.set_title(f'Spectrum: Under-sampled\n(Aliasing present)')
ax4.axvline(fs_under/2, color='k', linestyle='--', alpha=0.5, label='Nyquist freq')
ax4.legend()
ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(gs[1, 1])
ax5.stem(freqs_nyquist[:n_nyquist//2], np.abs(X_nyquist[:n_nyquist//2]), linefmt='g-', markerfmt='go', basefmt='k-')
ax5.set_xlabel('Frequency (Hz)')
ax5.set_ylabel('Magnitude')
ax5.set_title(f'Spectrum: Nyquist Sampling')
ax5.axvline(fs_nyquist/2, color='k', linestyle='--', alpha=0.5, label='Nyquist freq')
ax5.legend()
ax5.grid(True, alpha=0.3)

ax6 = fig.add_subplot(gs[1, 2])
ax6.stem(freqs_over[:n_over//2], np.abs(X_over[:n_over//2]), linefmt='m-', markerfmt='mo', basefmt='k-')
ax6.set_xlabel('Frequency (Hz)')
ax6.set_ylabel('Magnitude')
ax6.set_title(f'Spectrum: Over-sampled\n(No aliasing)')
ax6.axvline(fs_over/2, color='k', linestyle='--', alpha=0.5, label='Nyquist freq')
ax6.legend()
ax6.grid(True, alpha=0.3)
ax6.set_xlim([0, 150])

# Row 3: Aliasing visualization
ax7 = fig.add_subplot(gs[2, :2])
t_dense = np.linspace(0, 0.5, 5000, endpoint=False)
x_dense = np.sin(2*np.pi*f_alias_test*t_dense)
ax7.plot(t_dense, x_dense, 'b-', linewidth=2, alpha=0.5, label=f'Original ({f_alias_test} Hz)')
ax7.stem(t_alias, x_alias_cont, linefmt='r-', markerfmt='ro', basefmt='k-', label=f'Sampled at {fs_alias} Hz')
# Show the aliased frequency
x_aliased = np.sin(2*np.pi*f_apparent*t_dense)
ax7.plot(t_dense, x_aliased, 'g--', linewidth=2, label=f'Aliased ({f_apparent} Hz)')
ax7.set_xlabel('Time (s)')
ax7.set_ylabel('Amplitude')
ax7.set_title(f'Aliasing Demo: {f_alias_test} Hz sampled at {fs_alias} Hz appears as {f_apparent} Hz')
ax7.legend()
ax7.grid(True, alpha=0.3)
ax7.set_xlim([0, 0.2])

# Aliasing diagram
ax8 = fig.add_subplot(gs[2, 2])
frequencies = np.array([20, 50, 100, 150])
sampled_freqs = frequencies.copy()
for i, f in enumerate(frequencies):
    if f > fs_alias/2:
        # Calculate aliased frequency
        k = int(f / fs_alias)
        f_alias = f - k*fs_alias
        if f_alias > fs_alias/2:
            f_alias = fs_alias - f_alias
        sampled_freqs[i] = f_alias

ax8.scatter(frequencies, frequencies, s=100, c='blue', marker='o', label='True Frequency')
ax8.scatter(frequencies, sampled_freqs, s=100, c='red', marker='x', label='Sampled Frequency')
for i in range(len(frequencies)):
    ax8.plot([frequencies[i], frequencies[i]], [frequencies[i], sampled_freqs[i]], 
             'k--', alpha=0.3)
ax8.axhline(fs_alias/2, color='g', linestyle='--', linewidth=2, label=f'Nyquist ({fs_alias/2} Hz)')
ax8.set_xlabel('True Frequency (Hz)')
ax8.set_ylabel('Apparent Frequency (Hz)')
ax8.set_title('Frequency Aliasing Map')
ax8.legend()
ax8.grid(True, alpha=0.3)

# Row 4: Reconstruction methods comparison
ax9 = fig.add_subplot(gs[3, :2])
# Zero-order hold (nearest neighbor)
f_zoh = interpolate.interp1d(t_over, x_over, kind='nearest', fill_value='extrapolate')
x_zoh = f_zoh(t_recon)

# First-order hold (linear interpolation)
f_linear = interpolate.interp1d(t_over, x_over, kind='linear', fill_value='extrapolate')
x_linear = f_linear(t_recon)

# Cubic spline interpolation
f_cubic = interpolate.interp1d(t_over, x_over, kind='cubic', fill_value='extrapolate')
x_cubic = f_cubic(t_recon)

ax9.plot(t_continuous[:1000], x_continuous[:1000], 'b-', linewidth=2, alpha=0.5, label='Original')
ax9.plot(t_recon, x_zoh, 'r-', linewidth=1, alpha=0.7, label='Zero-order Hold')
ax9.plot(t_recon, x_linear, 'g-', linewidth=1, alpha=0.7, label='Linear Interpolation')
ax9.plot(t_recon, x_cubic, 'm-', linewidth=1, alpha=0.7, label='Cubic Spline')
ax9.stem(t_over[:50], x_over[:50], linefmt='k-', markerfmt='ko', basefmt='k-', 
         label='Samples', markerfacecolor='none')
ax9.set_xlabel('Time (s)')
ax9.set_ylabel('Amplitude')
ax9.set_title('Reconstruction Methods Comparison')
ax9.legend(loc='upper right')
ax9.grid(True, alpha=0.3)
ax9.set_xlim([0, 0.1])

# Reconstruction error analysis
ax10 = fig.add_subplot(gs[3, 2])
errors = {
    'Zero-order': np.sqrt(np.mean((x_zoh - x_continuous[:1000])**2)),
    'Linear': np.sqrt(np.mean((x_linear - x_continuous[:1000])**2)),
    'Cubic': np.sqrt(np.mean((x_cubic - x_continuous[:1000])**2)),
    'Sinc': np.sqrt(np.mean((x_recon_over - x_continuous[:1000])**2))
}
ax10.barh(list(errors.keys()), list(errors.values()), color=['red', 'green', 'magenta', 'cyan'])
ax10.set_xlabel('RMSE')
ax10.set_title('Reconstruction Error Comparison')
ax10.grid(True, alpha=0.3, axis='x')

plt.savefig('sampling_theory_comprehensive.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sampling Theory Summary:")
print(f"1. Signal frequency: {f_signal} Hz")
print(f"2. Nyquist rate: {f_nyquist} Hz (2 × signal frequency)")
print(f"3. Under-sampling: {fs_under} Hz → Aliasing occurs")
print(f"4. Nyquist sampling: {fs_nyquist} Hz → Minimum rate")
print(f"5. Over-sampling: {fs_over} Hz → Better reconstruction")
print(f"6. Aliasing example: {f_alias_test} Hz sampled at {fs_alias} Hz → appears as {f_apparent} Hz")
print(f"7. Reconstruction errors (RMSE):")
for method, error in errors.items():
    print(f"   - {method}: {error:.6f}")
print(f"8. Shannon-Nyquist Theorem: fs ≥ 2 × f_max for perfect reconstruction")

### 5.5 Sampling Theory: Nyquist Theorem, Aliasing, and Reconstruction

Understanding sampling is crucial for converting between continuous and discrete signals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq

# Create complex non-stationary test signals
fs = 8000  # High sampling rate for audio-like signals
duration = 4.0
t = np.linspace(0, duration, int(fs * duration), endpoint=False)

# Signal 1: Chirp (frequency sweep)
chirp = signal.chirp(t, f0=100, f1=2000, t1=duration, method='linear')

# Signal 2: Multiple tones appearing at different times
multitone = np.zeros_like(t)
multitone[(t >= 0) & (t < 1)] += 2*np.sin(2*np.pi*440*t[(t >= 0) & (t < 1)])     # A4 note
multitone[(t >= 1) & (t < 2)] += 2*np.sin(2*np.pi*554*t[(t >= 1) & (t < 2)])     # C#5 note
multitone[(t >= 2) & (t < 3)] += 2*np.sin(2*np.pi*659*t[(t >= 2) & (t < 3)])     # E5 note
multitone[(t >= 3) & (t < 4)] += (np.sin(2*np.pi*440*t[(t >= 3) & (t < 4)]) + 
                                   np.sin(2*np.pi*554*t[(t >= 3) & (t < 4)]) + 
                                   np.sin(2*np.pi*659*t[(t >= 3) & (t < 4)]))     # Chord

# Add noise
multitone += 0.2 * np.random.randn(len(t))

# Signal 3: Amplitude modulated signal
carrier_freq = 1000
mod_freq = 50
am_signal = (1 + 0.5*np.sin(2*np.pi*mod_freq*t)) * np.sin(2*np.pi*carrier_freq*t)

# Compute STFT with different window sizes
nperseg_short = 256   # Short window - good time resolution
nperseg_medium = 1024  # Medium window - balanced
nperseg_long = 4096   # Long window - good frequency resolution

# STFT for chirp signal
f_short, t_short, Zxx_short = signal.stft(chirp, fs, nperseg=nperseg_short, noverlap=nperseg_short//2)
f_medium, t_medium, Zxx_medium = signal.stft(chirp, fs, nperseg=nperseg_medium, noverlap=nperseg_medium//2)
f_long, t_long, Zxx_long = signal.stft(chirp, fs, nperseg=nperseg_long, noverlap=nperseg_long//2)

# STFT for multitone signal
f_mt, t_mt, Zxx_mt = signal.stft(multitone, fs, nperseg=nperseg_medium, noverlap=nperseg_medium//2)

# STFT for AM signal
f_am, t_am, Zxx_am = signal.stft(am_signal, fs, nperseg=nperseg_medium, noverlap=nperseg_medium//2)

# Create comprehensive visualization
fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)

# Row 1: Chirp signal with different window sizes
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t[:8000], chirp[:8000], 'b-', linewidth=1)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.set_title('Chirp Signal (100 → 2000 Hz)')
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.pcolormesh(t_short, f_short, np.abs(Zxx_short), shading='gouraud', cmap='viridis')
ax2.set_ylabel('Frequency (Hz)')
ax2.set_xlabel('Time (s)')
ax2.set_title(f'STFT - Short Window ({nperseg_short} samples)\nGood Time Resolution')
ax2.set_ylim([0, 2500])

ax3 = fig.add_subplot(gs[0, 2])
ax3.pcolormesh(t_long, f_long, np.abs(Zxx_long), shading='gouraud', cmap='viridis')
ax3.set_ylabel('Frequency (Hz)')
ax3.set_xlabel('Time (s)')
ax3.set_title(f'STFT - Long Window ({nperseg_long} samples)\nGood Frequency Resolution')
ax3.set_ylim([0, 2500])

# Row 2: Multitone signal
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(t, multitone, 'g-', linewidth=0.5, alpha=0.7)
ax4.set_xlabel('Time (s)')
ax4.set_ylabel('Amplitude')
ax4.set_title('Musical Notes Sequence\n(A4 → C#5 → E5 → Chord)')
ax4.grid(True, alpha=0.3)
ax4.axvline(1, color='r', linestyle='--', alpha=0.5)
ax4.axvline(2, color='r', linestyle='--', alpha=0.5)
ax4.axvline(3, color='r', linestyle='--', alpha=0.5)

ax5 = fig.add_subplot(gs[1, 1:])
spectrogram = ax5.pcolormesh(t_mt, f_mt, 20*np.log10(np.abs(Zxx_mt) + 1e-10), 
                              shading='gouraud', cmap='jet', vmin=-40, vmax=40)
ax5.set_ylabel('Frequency (Hz)')
ax5.set_xlabel('Time (s)')
ax5.set_title('Spectrogram - Musical Note Transitions')
ax5.set_ylim([0, 1500])
ax5.axhline(440, color='w', linestyle='--', alpha=0.5, linewidth=1)
ax5.axhline(554, color='w', linestyle='--', alpha=0.5, linewidth=1)
ax5.axhline(659, color='w', linestyle='--', alpha=0.5, linewidth=1)
plt.colorbar(spectrogram, ax=ax5, label='Power (dB)')

# Row 3: AM signal
ax6 = fig.add_subplot(gs[2, 0])
ax6.plot(t[:1000], am_signal[:1000], 'r-', linewidth=1)
ax6.set_xlabel('Time (s)')
ax6.set_ylabel('Amplitude')
ax6.set_title(f'AM Signal (Carrier: {carrier_freq} Hz, Mod: {mod_freq} Hz)')
ax6.grid(True, alpha=0.3)

ax7 = fig.add_subplot(gs[2, 1:])
spectrogram_am = ax7.pcolormesh(t_am, f_am, 20*np.log10(np.abs(Zxx_am) + 1e-10), 
                                 shading='gouraud', cmap='jet')
ax7.set_ylabel('Frequency (Hz)')
ax7.set_xlabel('Time (s)')
ax7.set_title('Spectrogram - Amplitude Modulation (Sidebands visible)')
ax7.set_ylim([0, 1500])
ax7.axhline(carrier_freq, color='w', linestyle='--', alpha=0.5, linewidth=1, label='Carrier')
ax7.axhline(carrier_freq - mod_freq, color='y', linestyle='--', alpha=0.5, linewidth=1, label='Lower sideband')
ax7.axhline(carrier_freq + mod_freq, color='c', linestyle='--', alpha=0.5, linewidth=1, label='Upper sideband')
ax7.legend(loc='upper right', fontsize=8)
plt.colorbar(spectrogram_am, ax=ax7, label='Power (dB)')

# Row 4: Comparison of different window functions
ax8 = fig.add_subplot(gs[3, 0])
windows = {
    'Rectangular': np.ones(nperseg_medium),
    'Hanning': np.hanning(nperseg_medium),
    'Hamming': np.hamming(nperseg_medium),
    'Blackman': np.blackman(nperseg_medium)
}
for name, win in windows.items():
    ax8.plot(win, label=name, linewidth=2, alpha=0.7)
ax8.set_xlabel('Sample')
ax8.set_ylabel('Amplitude')
ax8.set_title('Window Functions Comparison')
ax8.legend()
ax8.grid(True, alpha=0.3)

# Frequency response of windows
ax9 = fig.add_subplot(gs[3, 1])
for name, win in windows.items():
    W = fft(win, 2048)
    freqs = fftfreq(2048, 1/fs)
    ax9.plot(freqs[:1024], 20*np.log10(np.abs(W[:1024]) + 1e-10), label=name, linewidth=2, alpha=0.7)
ax9.set_xlabel('Frequency (Hz)')
ax9.set_ylabel('Magnitude (dB)')
ax9.set_title('Window Frequency Responses')
ax9.set_xlim([0, 200])
ax9.set_ylim([-100, 40])
ax9.legend()
ax9.grid(True, alpha=0.3)

# Time-frequency resolution trade-off
ax10 = fig.add_subplot(gs[3, 2])
window_sizes = [128, 256, 512, 1024, 2048, 4096]
time_res = [ws/fs*1000 for ws in window_sizes]  # in ms
freq_res = [fs/ws for ws in window_sizes]  # in Hz
ax10.loglog(time_res, freq_res, 'bo-', linewidth=2, markersize=8)
ax10.set_xlabel('Time Resolution (ms)')
ax10.set_ylabel('Frequency Resolution (Hz)')
ax10.set_title('Time-Frequency Resolution Trade-off')
ax10.grid(True, alpha=0.3, which='both')
for i, ws in enumerate(window_sizes):
    ax10.annotate(f'{ws}', (time_res[i], freq_res[i]), 
                  textcoords="offset points", xytext=(5,5), fontsize=8)

plt.savefig('time_frequency_analysis_comprehensive.png', dpi=150, bbox_inches='tight')
plt.show()

# Print analysis summary
print("Time-Frequency Analysis Summary:")
print(f"1. Sampling Frequency: {fs} Hz")
print(f"2. Signal Duration: {duration} s")
print(f"3. Short window: {nperseg_short} samples = {nperseg_short/fs*1000:.2f} ms")
print(f"   - Time resolution: {nperseg_short/fs*1000:.2f} ms")
print(f"   - Frequency resolution: {fs/nperseg_short:.2f} Hz")
print(f"4. Long window: {nperseg_long} samples = {nperseg_long/fs*1000:.2f} ms")
print(f"   - Time resolution: {nperseg_long/fs*1000:.2f} ms")
print(f"   - Frequency resolution: {fs/nperseg_long:.2f} Hz")
print(f"5. Uncertainty principle: Δt × Δf ≥ 1/(4π)")
print(f"6. Musical notes: A4 (440 Hz), C#5 (554 Hz), E5 (659 Hz)")
print(f"7. AM sidebands: {carrier_freq-mod_freq} Hz, {carrier_freq+mod_freq} Hz")

### 5.4 Time-Frequency Analysis: STFT and Spectrograms

Short-Time Fourier Transform provides time-localized frequency analysis for non-stationary signals.

In [ ]:
import pywt
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# Create a non-stationary test signal with changing frequencies
fs = 1000
t = np.linspace(0, 4, 4*fs, endpoint=False)

# Piecewise signal: different frequencies in different time segments
x = np.zeros_like(t)
x[(t >= 0) & (t < 1)] = np.sin(2*np.pi*10*t[(t >= 0) & (t < 1)])     # 10 Hz
x[(t >= 1) & (t < 2)] = np.sin(2*np.pi*50*t[(t >= 1) & (t < 2)])     # 50 Hz
x[(t >= 2) & (t < 3)] = np.sin(2*np.pi*100*t[(t >= 2) & (t < 3)])    # 100 Hz
x[(t >= 3) & (t < 4)] = signal.chirp(t[(t >= 3) & (t < 4)]-3, 10, 1, 100)  # Chirp

# Add some noise
x += 0.1 * np.random.randn(len(t))

# Perform Discrete Wavelet Transform (DWT)
wavelet = 'db4'  # Daubechies 4 wavelet
level = 5  # Decomposition level

coeffs = pywt.wavedec(x, wavelet, level=level)
cA5, cD5, cD4, cD3, cD2, cD1 = coeffs

# Perform Continuous Wavelet Transform (CWT) for time-frequency analysis
scales = np.arange(1, 128)
wavelet_cwt = 'morl'  # Morlet wavelet
coefficients, frequencies = pywt.cwt(x[::4], scales, wavelet_cwt, 1/fs*4)  # Downsample for speed

# Wavelet packet decomposition for detailed multi-resolution analysis
wp = pywt.WaveletPacket(data=x, wavelet=wavelet, mode='symmetric', maxlevel=3)

# Create comprehensive visualization
fig = plt.figure(figsize=(16, 14))
gs = fig.add_gridspec(5, 3, hspace=0.4, wspace=0.3)

# Original signal
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(t, x, 'b-', linewidth=1)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.set_title('Original Non-Stationary Signal (10 Hz → 50 Hz → 100 Hz → Chirp)')
ax1.grid(True, alpha=0.3)
ax1.axvline(1, color='r', linestyle='--', alpha=0.5)
ax1.axvline(2, color='r', linestyle='--', alpha=0.5)
ax1.axvline(3, color='r', linestyle='--', alpha=0.5)

# Discrete Wavelet Transform decomposition
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(cA5, 'g-', linewidth=1)
ax2.set_title(f'Approximation (cA{level}) - Low Freq')
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(cD5, 'r-', linewidth=1)
ax3.set_title(f'Detail (cD{level}) - Highest Freq')
ax3.grid(True, alpha=0.3)

ax4 = fig.add_subplot(gs[1, 2])
ax4.plot(cD4, 'm-', linewidth=1)
ax4.set_title(f'Detail (cD{level-1})')
ax4.grid(True, alpha=0.3)

ax5 = fig.add_subplot(gs[2, 0])
ax5.plot(cD3, 'c-', linewidth=1)
ax5.set_title(f'Detail (cD{level-2})')
ax5.grid(True, alpha=0.3)

ax6 = fig.add_subplot(gs[2, 1])
ax6.plot(cD2, 'orange', linewidth=1)
ax6.set_title(f'Detail (cD{level-3})')
ax6.grid(True, alpha=0.3)

ax7 = fig.add_subplot(gs[2, 2])
ax7.plot(cD1, 'brown', linewidth=1)
ax7.set_title(f'Detail (cD{level-4}) - Lowest Freq')
ax7.grid(True, alpha=0.3)

# Continuous Wavelet Transform - Scalogram (time-frequency map)
ax8 = fig.add_subplot(gs[3, :2])
im = ax8.imshow(np.abs(coefficients), extent=[0, 4, frequencies[-1], frequencies[0]], 
                cmap='jet', aspect='auto', interpolation='bilinear')
ax8.set_xlabel('Time (s)')
ax8.set_ylabel('Frequency (Hz)')
ax8.set_title('Continuous Wavelet Transform - Scalogram (Time-Frequency Map)')
ax8.axvline(1, color='w', linestyle='--', alpha=0.5)
ax8.axvline(2, color='w', linestyle='--', alpha=0.5)
ax8.axvline(3, color='w', linestyle='--', alpha=0.5)
plt.colorbar(im, ax=ax8, label='|CWT Coefficient|')

# Wavelet basis functions
ax9 = fig.add_subplot(gs[3, 2])
for i, wavelet_name in enumerate(['db4', 'sym4', 'coif2', 'bior2.4']):
    psi, x_psi = pywt.Wavelet(wavelet_name).wavfun(level=8)
    ax9.plot(x_psi, psi + i*2, linewidth=2, label=wavelet_name)
ax9.set_xlabel('Time')
ax9.set_ylabel('Amplitude (offset)')
ax9.set_title('Different Wavelet Basis Functions')
ax9.legend()
ax9.grid(True, alpha=0.3)

# Denoising using wavelet thresholding
# Apply soft thresholding to detail coefficients
coeffs_thresh = coeffs.copy()
sigma = np.median(np.abs(cD1)) / 0.6745  # Noise estimation
threshold = sigma * np.sqrt(2 * np.log(len(x)))

for i in range(1, len(coeffs_thresh)):
    coeffs_thresh[i] = pywt.threshold(coeffs_thresh[i], threshold, mode='soft')

x_denoised = pywt.waverec(coeffs_thresh, wavelet)

# Trim to original length if needed
if len(x_denoised) > len(x):
    x_denoised = x_denoised[:len(x)]

ax10 = fig.add_subplot(gs[4, :2])
ax10.plot(t, x, 'b-', alpha=0.5, linewidth=1, label='Noisy Signal')
ax10.plot(t, x_denoised, 'r-', linewidth=2, label='Denoised (Wavelet Thresholding)')
ax10.set_xlabel('Time (s)')
ax10.set_ylabel('Amplitude')
ax10.set_title('Wavelet-based Denoising')
ax10.legend()
ax10.grid(True, alpha=0.3)

# Energy distribution across scales
ax11 = fig.add_subplot(gs[4, 2])
energies = [np.sum(c**2) for c in coeffs]
labels = [f'cA{level}'] + [f'cD{level-i}' for i in range(level)]
ax11.bar(labels, energies, color=['green'] + ['red', 'magenta', 'cyan', 'orange', 'brown'])
ax11.set_ylabel('Energy')
ax11.set_title('Energy Distribution Across Scales')
ax11.grid(True, alpha=0.3, axis='y')
ax11.tick_params(axis='x', rotation=45)

plt.savefig('wavelet_analysis_comprehensive.png', dpi=150, bbox_inches='tight')
plt.show()

# Print wavelet properties
print("Wavelet Analysis Summary:")
print(f"1. Wavelet Family: {wavelet}")
print(f"2. Decomposition Levels: {level}")
print(f"3. Original Signal Length: {len(x)}")
print(f"4. Approximation Length (level {level}): {len(cA5)}")
print(f"5. Total Energy (original): {np.sum(x**2):.2f}")
print(f"6. Total Energy (reconstructed): {sum(energies):.2f}")
print(f"7. Denoising threshold: {threshold:.4f}")
print(f"8. SNR improvement: {10*np.log10(np.var(x_denoised) / np.var(x - x_denoised)):.2f} dB")
print(f"9. CWT Scales analyzed: {len(scales)}")
print(f"10. Frequency range: {frequencies[-1]:.1f} - {frequencies[0]:.1f} Hz")

### 5.3 Wavelet Analysis and Multi-Resolution Decomposition

Wavelets provide localized time-frequency analysis, ideal for non-stationary signals and multi-scale analysis.

In [ ]:
from scipy import signal
import numpy as np
import matplotlib.pyplot as plt

# Generate a test signal with multiple frequency components
fs = 1000  # Sampling frequency
T = 2.0
N = int(fs * T)
t = np.linspace(0, T, N, endpoint=False)

# Signal: Low freq (5 Hz) + Mid freq (50 Hz) + High freq (200 Hz) + noise
x = (2.0 * np.sin(2*np.pi*5*t) +       # Low frequency component
     1.0 * np.sin(2*np.pi*50*t) +      # Mid frequency component
     0.5 * np.sin(2*np.pi*200*t) +     # High frequency component
     0.3 * np.random.randn(N))         # White noise

# Design different types of filters using Butterworth design
order = 4  # Filter order

# 1. Low-pass filter (cutoff at 30 Hz)
lowpass_b, lowpass_a = signal.butter(order, 30, btype='low', fs=fs)
x_lowpass = signal.filtfilt(lowpass_b, lowpass_a, x)

# 2. High-pass filter (cutoff at 100 Hz)
highpass_b, highpass_a = signal.butter(order, 100, btype='high', fs=fs)
x_highpass = signal.filtfilt(highpass_b, highpass_a, x)

# 3. Band-pass filter (passband: 40-60 Hz)
bandpass_b, bandpass_a = signal.butter(order, [40, 60], btype='band', fs=fs)
x_bandpass = signal.filtfilt(bandpass_b, bandpass_a, x)

# 4. Band-stop (notch) filter (stopband: 45-55 Hz)
bandstop_b, bandstop_a = signal.butter(order, [45, 55], btype='bandstop', fs=fs)
x_bandstop = signal.filtfilt(bandstop_b, bandstop_a, x)

# Create comprehensive visualization
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)

# Plot time domain signals
time_slice = slice(0, 500)  # Show first 0.5 seconds

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t[time_slice], x[time_slice], 'b-', linewidth=1, alpha=0.7)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.set_title('Original Signal (5 + 50 + 200 Hz + noise)')
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(t[time_slice], x_lowpass[time_slice], 'g-', linewidth=1.5)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Amplitude')
ax2.set_title('Low-pass Filtered (< 30 Hz)')
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[2, 0])
ax3.plot(t[time_slice], x_highpass[time_slice], 'r-', linewidth=1.5)
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Amplitude')
ax3.set_title('High-pass Filtered (> 100 Hz)')
ax3.grid(True, alpha=0.3)

ax4 = fig.add_subplot(gs[3, 0])
ax4.plot(t[time_slice], x_bandpass[time_slice], 'm-', linewidth=1.5)
ax4.set_xlabel('Time (s)')
ax4.set_ylabel('Amplitude')
ax4.set_title('Band-pass Filtered (40-60 Hz)')
ax4.grid(True, alpha=0.3)

# Plot frequency responses
w_lp, h_lp = signal.freqz(lowpass_b, lowpass_a, worN=2048, fs=fs)
w_hp, h_hp = signal.freqz(highpass_b, highpass_a, worN=2048, fs=fs)
w_bp, h_bp = signal.freqz(bandpass_b, bandpass_a, worN=2048, fs=fs)
w_bs, h_bs = signal.freqz(bandstop_b, bandstop_a, worN=2048, fs=fs)

ax5 = fig.add_subplot(gs[0, 1])
ax5.plot(w_lp, 20*np.log10(np.abs(h_lp)), 'g-', linewidth=2, label='Low-pass')
ax5.plot(w_hp, 20*np.log10(np.abs(h_hp)), 'r-', linewidth=2, label='High-pass')
ax5.set_xlabel('Frequency (Hz)')
ax5.set_ylabel('Magnitude (dB)')
ax5.set_title('Filter Frequency Responses (Magnitude)')
ax5.set_xlim([0, 300])
ax5.set_ylim([-80, 5])
ax5.legend()
ax5.grid(True, alpha=0.3)
ax5.axhline(-3, color='k', linestyle='--', alpha=0.5)

ax6 = fig.add_subplot(gs[1, 1])
ax6.plot(w_bp, 20*np.log10(np.abs(h_bp)), 'm-', linewidth=2, label='Band-pass')
ax6.plot(w_bs, 20*np.log10(np.abs(h_bs)), 'c-', linewidth=2, label='Band-stop')
ax6.set_xlabel('Frequency (Hz)')
ax6.set_ylabel('Magnitude (dB)')
ax6.set_title('Band Filters Frequency Responses')
ax6.set_xlim([0, 100])
ax6.set_ylim([-80, 5])
ax6.legend()
ax6.grid(True, alpha=0.3)
ax6.axhline(-3, color='k', linestyle='--', alpha=0.5)

# Plot spectra of filtered signals
from scipy.fft import fft, fftfreq

freqs = fftfreq(N, 1/fs)
X_orig = np.abs(fft(x))
X_lowpass = np.abs(fft(x_lowpass))
X_highpass = np.abs(fft(x_highpass))
X_bandpass = np.abs(fft(x_bandpass))

ax7 = fig.add_subplot(gs[2, 1])
ax7.semilogy(freqs[:N//2], X_orig[:N//2], 'b-', alpha=0.5, linewidth=1, label='Original')
ax7.semilogy(freqs[:N//2], X_lowpass[:N//2], 'g-', linewidth=2, label='Low-pass')
ax7.set_xlabel('Frequency (Hz)')
ax7.set_ylabel('Magnitude (log scale)')
ax7.set_title('Spectrum Comparison: Original vs Low-pass')
ax7.set_xlim([0, 300])
ax7.legend()
ax7.grid(True, alpha=0.3)

ax8 = fig.add_subplot(gs[3, 1])
ax8.semilogy(freqs[:N//2], X_orig[:N//2], 'b-', alpha=0.5, linewidth=1, label='Original')
ax8.semilogy(freqs[:N//2], X_bandpass[:N//2], 'm-', linewidth=2, label='Band-pass')
ax8.set_xlabel('Frequency (Hz)')
ax8.set_ylabel('Magnitude (log scale)')
ax8.set_title('Spectrum Comparison: Original vs Band-pass')
ax8.set_xlim([0, 100])
ax8.legend()
ax8.grid(True, alpha=0.3)

# Pole-zero plots for different filters
ax9 = fig.add_subplot(gs[0, 2])
zeros, poles, _ = signal.tf2zpk(lowpass_b, lowpass_a)
ax9.scatter(np.real(zeros), np.imag(zeros), s=50, marker='o', color='g', label='Zeros')
ax9.scatter(np.real(poles), np.imag(poles), s=50, marker='x', color='r', label='Poles')
theta = np.linspace(0, 2*np.pi, 100)
ax9.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3)
ax9.set_xlabel('Real')
ax9.set_ylabel('Imaginary')
ax9.set_title('Pole-Zero Plot: Low-pass Filter')
ax9.legend()
ax9.grid(True, alpha=0.3)
ax9.axis('equal')
ax9.set_xlim([-1.5, 1.5])
ax9.set_ylim([-1.5, 1.5])

# Impulse response
ax10 = fig.add_subplot(gs[1, 2])
_, h_imp_lp = signal.impulse((lowpass_b, lowpass_a), N=100, T=np.linspace(0, 0.1, 100))
_, h_imp_hp = signal.impulse((highpass_b, highpass_a), N=100, T=np.linspace(0, 0.1, 100))
ax10.plot(h_imp_lp, 'g-', linewidth=2, label='Low-pass')
ax10.plot(h_imp_hp, 'r-', linewidth=2, label='High-pass')
ax10.set_xlabel('Sample')
ax10.set_ylabel('Amplitude')
ax10.set_title('Impulse Responses')
ax10.legend()
ax10.grid(True, alpha=0.3)

# Step response
ax11 = fig.add_subplot(gs[2, 2])
_, s_step_lp = signal.step((lowpass_b, lowpass_a), N=100, T=np.linspace(0, 0.1, 100))
_, s_step_bp = signal.step((bandpass_b, bandpass_a), N=100, T=np.linspace(0, 0.1, 100))
ax11.plot(s_step_lp, 'g-', linewidth=2, label='Low-pass')
ax11.plot(s_step_bp, 'm-', linewidth=2, label='Band-pass')
ax11.set_xlabel('Sample')
ax11.set_ylabel('Amplitude')
ax11.set_title('Step Responses')
ax11.legend()
ax11.grid(True, alpha=0.3)

# Filter comparison metrics
ax12 = fig.add_subplot(gs[3, 2])
filters = ['Low-pass', 'High-pass', 'Band-pass', 'Band-stop']
snr_improvements = []
for filtered_sig in [x_lowpass, x_highpass, x_bandpass, x_bandstop]:
    # Simple SNR estimate
    signal_power = np.var(filtered_sig)
    noise_power = np.var(x - filtered_sig)
    snr = 10 * np.log10(signal_power / (noise_power + 1e-10))
    snr_improvements.append(snr)

ax12.barh(filters, snr_improvements, color=['green', 'red', 'magenta', 'cyan'])
ax12.set_xlabel('SNR (dB)')
ax12.set_title('Filter Performance Comparison')
ax12.grid(True, alpha=0.3, axis='x')

plt.savefig('digital_filtering_comprehensive.png', dpi=150, bbox_inches='tight')
plt.show()

print("Filter Design Summary:")
print(f"1. Sampling Frequency: {fs} Hz")
print(f"2. Filter Order: {order}")
print(f"3. Low-pass cutoff: 30 Hz (-3dB point)")
print(f"4. High-pass cutoff: 100 Hz (-3dB point)")
print(f"5. Band-pass range: 40-60 Hz")
print(f"6. Band-stop range: 45-55 Hz")
print(f"7. All filters use zero-phase filtering (filtfilt) for linear phase response")

### 5.2 Digital Filtering: Low-pass, High-pass, Band-pass, and Band-stop Filters

Filters are essential for signal conditioning, noise removal, and feature extraction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq, fftshift
from scipy import signal

# Create a composite signal with multiple frequency components
fs = 1000  # Sampling frequency (Hz)
T = 2.0    # Duration (seconds)
N = int(fs * T)  # Number of samples
t = np.linspace(0, T, N, endpoint=False)

# Composite signal: sum of sinusoids with different frequencies
f1, f2, f3 = 5, 20, 50  # Frequencies in Hz
A1, A2, A3 = 1.0, 0.5, 0.3  # Amplitudes
x = A1 * np.sin(2*np.pi*f1*t) + A2 * np.sin(2*np.pi*f2*t) + A3 * np.sin(2*np.pi*f3*t)

# Add some noise
noise = 0.1 * np.random.randn(N)
x_noisy = x + noise

# Compute DFT using FFT
X = fft(x_noisy)
freqs = fftfreq(N, 1/fs)

# Compute magnitude and phase
magnitude = np.abs(X)
phase = np.angle(X)

# Create comprehensive visualization
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# Time domain signal
axes[0, 0].plot(t[:500], x[:500], 'b-', linewidth=2, label='Clean Signal')
axes[0, 0].plot(t[:500], x_noisy[:500], 'r-', alpha=0.5, label='Noisy Signal')
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].set_title('Time Domain Signal')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Frequency domain - magnitude spectrum
axes[0, 1].plot(freqs[:N//2], magnitude[:N//2], 'b-', linewidth=1.5)
axes[0, 1].set_xlabel('Frequency (Hz)')
axes[0, 1].set_ylabel('Magnitude')
axes[0, 1].set_title('Magnitude Spectrum (DFT)')
axes[0, 1].set_xlim([0, 100])
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axvline(f1, color='r', linestyle='--', alpha=0.5, label=f'{f1} Hz')
axes[0, 1].axvline(f2, color='g', linestyle='--', alpha=0.5, label=f'{f2} Hz')
axes[0, 1].axvline(f3, color='m', linestyle='--', alpha=0.5, label=f'{f3} Hz')
axes[0, 1].legend()

# Phase spectrum
axes[1, 0].plot(freqs[:N//2], phase[:N//2], 'g-', linewidth=1)
axes[1, 0].set_xlabel('Frequency (Hz)')
axes[1, 0].set_ylabel('Phase (radians)')
axes[1, 0].set_title('Phase Spectrum')
axes[1, 0].set_xlim([0, 100])
axes[1, 0].grid(True, alpha=0.3)

# Parseval's Theorem: Energy conservation
time_energy = np.sum(x_noisy**2) / N
freq_energy = np.sum(np.abs(X)**2) / N**2
axes[1, 1].bar(['Time Domain', 'Frequency Domain'], [time_energy, freq_energy], 
               color=['blue', 'red'], alpha=0.7)
axes[1, 1].set_ylabel('Energy')
axes[1, 1].set_title(f"Parseval's Theorem\nError: {abs(time_energy - freq_energy):.2e}")
axes[1, 1].grid(True, alpha=0.3, axis='y')

# Inverse FFT - Signal reconstruction
X_filtered = X.copy()
# Zero out high frequencies to demonstrate filtering
X_filtered[np.abs(freqs) > 60] = 0
x_reconstructed = np.real(ifft(X_filtered))

axes[2, 0].plot(t[:500], x_noisy[:500], 'r-', alpha=0.5, label='Noisy Signal')
axes[2, 0].plot(t[:500], x_reconstructed[:500], 'b-', linewidth=2, label='Reconstructed (Low-pass)')
axes[2, 0].set_xlabel('Time (s)')
axes[2, 0].set_ylabel('Amplitude')
axes[2, 0].set_title('Signal Reconstruction via Inverse FFT')
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)

# Convolution theorem: Multiplication in frequency domain = Convolution in time domain
# Create a simple filter kernel
kernel = np.hanning(21)
kernel = kernel / np.sum(kernel)  # Normalize

# Time domain convolution
y_conv_time = np.convolve(x_noisy, kernel, mode='same')

# Frequency domain multiplication
H = fft(kernel, N)  # Pad kernel to same length
Y_conv_freq = X * H
y_conv_freq = np.real(ifft(Y_conv_freq))

axes[2, 1].plot(t[:500], y_conv_time[:500], 'b-', linewidth=2, label='Time Domain Conv.')
axes[2, 1].plot(t[:500], y_conv_freq[:500], 'r--', linewidth=2, alpha=0.7, label='Freq Domain Conv.')
axes[2, 1].set_xlabel('Time (s)')
axes[2, 1].set_ylabel('Amplitude')
axes[2, 1].set_title('Convolution Theorem Verification')
axes[2, 1].legend()
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fourier_analysis_comprehensive.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key Properties Demonstrated:")
print(f"1. Frequency Resolution: Δf = {fs/N:.3f} Hz")
print(f"2. Parseval's Theorem - Energy Conservation Error: {abs(time_energy - freq_energy):.2e}")
print(f"3. Detected Peaks: {f1} Hz, {f2} Hz, {f3} Hz")
print(f"4. Convolution Theorem - Max difference: {np.max(np.abs(y_conv_time - y_conv_freq)):.2e}")
print(f"5. Nyquist Frequency: {fs/2} Hz")

### 5.1 Fourier Transform: DFT, FFT, Properties and Applications

Understanding the Discrete Fourier Transform and its efficient implementation via FFT is fundamental to signal processing.

## 5. Practical Demonstrations and Visualizations

The following cells provide hands-on demonstrations of key signal processing concepts with comprehensive visualizations.

## Comprehensive Summary: Advanced Signal Processing Framework

### Mathematical Foundations and Theoretical Synthesis

This comprehensive framework demonstrates the deep mathematical foundations underlying signal processing and their essential role in modern machine learning and AI systems. The implementation covers:

#### 1. **Mathematical Rigor and Signal Spaces**
- **Hilbert Space Theory**: Signal spaces as function spaces with inner product structure
- **Parseval's Theorem**: Energy conservation between time and frequency domains
- **Uncertainty Principle**: Fundamental trade-off between time and frequency resolution
- **Norm Theory**: L^p spaces and their applications in signal analysis

#### 2. **Advanced Transform Theory**
- **Fourier Analysis**: Complete framework from classical series to modern FFT implementations
- **Wavelet Theory**: Multiresolution analysis with perfect reconstruction
- **Z-Transform**: Complex analysis for discrete-time systems
- **Time-Frequency Analysis**: STFT, Gabor transforms, and spectrograms

#### 3. **Filter Design and System Theory**
- **Optimal Filter Design**: Parks-McClellan algorithm and equiripple approximation
- **IIR vs FIR Trade-offs**: Stability, phase response, and computational efficiency
- **Adaptive Algorithms**: LMS and RLS with convergence analysis
- **Multirate Processing**: Polyphase decomposition and efficient implementation

#### 4. **Statistical Signal Processing**
- **Spectral Estimation**: Non-parametric (Welch, multitaper) and parametric (AR, MUSIC) methods
- **Wiener Filtering**: Optimal linear filtering theory
- **Detection Theory**: Hypothesis testing and performance bounds
- **Estimation Theory**: Maximum likelihood and Bayesian approaches

#### 5. **Compressed Sensing Revolution**
- **Sparsity Theory**: Mathematical foundations of sparse signal recovery
- **RIP Theory**: Restricted Isometry Property and measurement matrix design
- **Recovery Algorithms**: L1 minimization, OMP, and iterative methods
- **Applications**: Medical imaging, radar, and sensor networks

#### 6. **Machine Learning Integration**
- **Feature Extraction**: Time-domain, frequency-domain, and time-frequency features
- **Blind Source Separation**: ICA and matrix factorization techniques
- **Convolutional Processing**: 1D convolutions as learnable filters
- **Graph Signal Processing**: Extension to non-Euclidean domains

#### 7. **Modern Paradigms**
- **Graph Signal Processing**: Signals on irregular structures
- **Manifold Processing**: Signal analysis on curved spaces
- **Quantum Signal Processing**: Emerging quantum computational approaches
- **Deep Learning Integration**: Differentiable signal processing pipelines

### Key Theoretical Insights

1. **Duality Principle**: Time and frequency domains provide complementary views of signals
2. **Trade-off Relationships**: Resolution, stability, and computational complexity trade-offs
3. **Optimality Theory**: Mathematical frameworks for optimal filter design
4. **Sparsity Paradigm**: Revolutionary approach to signal acquisition and processing
5. **Geometric Perspectives**: Viewing signals as points in high-dimensional spaces

### Practical Applications

The framework demonstrates practical implementations across:

- **Audio Processing**: Speech recognition, music analysis, noise reduction
- **Computer Vision**: Image filtering, feature detection, compression
- **Biomedical Engineering**: ECG/EEG analysis, medical imaging
- **Communications**: Channel coding, equalization, synchronization
- **Machine Learning**: Preprocessing, feature engineering, model design

### Advanced Mathematical Connections

Signal processing bridges multiple mathematical disciplines:

- **Functional Analysis**: Operators, spaces, and convergence theory
- **Harmonic Analysis**: Group theory and representation theory
- **Approximation Theory**: Best approximation in normed spaces
- **Probability Theory**: Stochastic processes and random signals
- **Optimization Theory**: Convex optimization and variational methods

### Research Frontiers

Current and emerging research directions include:

1. **Physics-Informed Signal Processing**: Incorporating physical laws and constraints
2. **Geometric Deep Learning**: Neural networks on graphs and manifolds
3. **Quantum Signal Processing**: Leveraging quantum superposition and entanglement
4. **Sparse and Low-Rank Models**: Advanced regularization techniques
5. **Federated Signal Processing**: Distributed learning and privacy preservation

### Implementation Excellence

The framework provides:

- **Mathematical Rigor**: Formal definitions, theorems, and proofs
- **Computational Efficiency**: Optimized algorithms and implementations
- **Practical Relevance**: Real-world applications and use cases
- **Educational Value**: Clear exposition from theory to practice
- **Research Foundation**: Starting point for advanced research

This comprehensive treatment establishes signal processing as a fundamental mathematical discipline essential for understanding and advancing machine learning, artificial intelligence, and modern data science applications.

# Signal Processing: Mathematical Foundations for Machine Learning and AI

## Table of Contents
1. [Introduction and Theoretical Foundations](#introduction)
2. [Fourier Analysis and Harmonic Theory](#fourier-analysis)
3. [Discrete-Time Signal Processing](#discrete-time)
4. [Digital Filter Design and Analysis](#digital-filtering)
5. [Time-Frequency Analysis and Wavelets](#time-frequency)
6. [Spectral Estimation and Analysis](#spectral-analysis)
7. [Multirate Signal Processing](#multirate)
8. [Adaptive Signal Processing](#adaptive)
9. [Statistical Signal Processing](#statistical)
10. [Compressed Sensing and Sparse Signals](#compressed-sensing)
11. [Machine Learning Applications](#ml-applications)
12. [Advanced Topics and Modern Methods](#advanced-topics)
13. [Summary and Theoretical Synthesis](#summary)

## 1. Introduction and Theoretical Foundations {#introduction}

Signal processing provides the mathematical framework for analyzing, transforming, and extracting information from signals—functions that vary with respect to one or more independent variables, typically time or space. As a cornerstone of modern engineering and machine learning, signal processing bridges continuous and discrete mathematics, providing essential tools for understanding and manipulating information-bearing waveforms.

### Historical Development and Mathematical Evolution

The development of signal processing theory evolved through several transformative periods:
- **Classical Harmonic Analysis**: Fourier's work on heat conduction (1807), establishing frequency domain analysis
- **Sampling Theory**: Nyquist-Shannon theorem (1928-1949), connecting continuous and discrete signals
- **Digital Revolution**: Fast Fourier Transform (Cooley-Tukey, 1965), enabling computational signal processing
- **Wavelet Theory**: Multi-resolution analysis (Daubechies, 1980s), time-frequency localization
- **Compressed Sensing**: Sparse signal recovery (Candès-Tao-Donoho, 2000s), sub-Nyquist sampling

### Mathematical Framework and Signal Spaces

**Definition 1.1** (Signal): A **signal** is a function $x: \mathbb{R}^n \to \mathbb{C}$ (or $x: \mathbb{Z}^n \to \mathbb{C}$ for discrete signals) that represents the variation of a physical quantity.

**Common Signal Classes**:
1. **$L^2$ Signals**: Energy signals with $\int_{-\infty}^{\infty} |x(t)|^2 dt < \infty$
2. **$L^1$ Signals**: Absolutely integrable with $\int_{-\infty}^{\infty} |x(t)| dt < \infty$  
3. **Periodic Signals**: $x(t + T) = x(t)$ for some fundamental period $T$
4. **Band-limited Signals**: Fourier transform $X(\omega) = 0$ for $|\omega| > W$
5. **Stationary Processes**: Statistical properties invariant under time shifts

### Fourier Analysis: The Frequency Domain

**Definition 1.2** (Continuous Fourier Transform): For signal $x(t) \in L^1(\mathbb{R})$:
$$X(\omega) = \mathcal{F}\{x(t)\} = \int_{-\infty}^{\infty} x(t) e^{-j\omega t} dt$$

**Inverse Transform**:
$$x(t) = \mathcal{F}^{-1}\{X(\omega)\} = \frac{1}{2\pi} \int_{-\infty}^{\infty} X(\omega) e^{j\omega t} d\omega$$

**Fundamental Properties**:
1. **Linearity**: $\mathcal{F}\{ax(t) + by(t)\} = aX(\omega) + bY(\omega)$
2. **Time Shift**: $\mathcal{F}\{x(t - t_0)\} = X(\omega)e^{-j\omega t_0}$
3. **Frequency Shift**: $\mathcal{F}\{x(t)e^{j\omega_0 t}\} = X(\omega - \omega_0)$
4. **Convolution**: $\mathcal{F}\{x(t) * y(t)\} = X(\omega)Y(\omega)$
5. **Parseval's Theorem**: $\int |x(t)|^2 dt = \frac{1}{2\pi} \int |X(\omega)|^2 d\omega$

### Sampling Theory and Discrete Signals

**Theorem 1.1** (Nyquist-Shannon Sampling Theorem): A band-limited signal $x(t)$ with bandwidth $W$ can be perfectly reconstructed from its samples $x(nT_s)$ if the sampling frequency satisfies:
$$f_s = \frac{1}{T_s} \geq 2W$$

**Reconstruction Formula**:
$$x(t) = \sum_{n=-\infty}^{\infty} x(nT_s) \frac{\sin(\pi(t - nT_s)/T_s)}{\pi(t - nT_s)/T_s}$$

**Definition 1.3** (Discrete Fourier Transform): For finite sequence $x[n], n = 0, 1, \ldots, N-1$:
$$X[k] = \sum_{n=0}^{N-1} x[n] e^{-j2\pi kn/N}, \quad k = 0, 1, \ldots, N-1$$

**Inverse DFT**:
$$x[n] = \frac{1}{N} \sum_{k=0}^{N-1} X[k] e^{j2\pi kn/N}$$

### Linear Time-Invariant Systems

**Definition 1.4** (LTI System): A system is **linear time-invariant** if:
1. **Linearity**: $T\{ax_1[n] + bx_2[n]\} = aT\{x_1[n]\} + bT\{x_2[n]\}$
2. **Time-Invariance**: $T\{x[n - n_0]\} = y[n - n_0]$ where $y[n] = T\{x[n]\}$

**Convolution Representation**: For LTI system with impulse response $h[n]$:
$$y[n] = x[n] * h[n] = \sum_{k=-\infty}^{\infty} x[k]h[n-k]$$

**Frequency Response**: $H(\omega) = \mathcal{F}\{h[n]\}$, giving $Y(\omega) = X(\omega)H(\omega)$.

### Digital Filter Theory

**Definition 1.5** (Digital Filter): A discrete-time LTI system characterized by difference equation:
$$\sum_{k=0}^N a_k y[n-k] = \sum_{k=0}^M b_k x[n-k]$$

**Transfer Function**:
$$H(z) = \frac{\sum_{k=0}^M b_k z^{-k}}{\sum_{k=0}^N a_k z^{-k}}$$

**Filter Classifications**:
1. **FIR (Finite Impulse Response)**: $a_k = 0$ for $k > 0$, always stable
2. **IIR (Infinite Impulse Response)**: $a_k \neq 0$ for some $k > 0$, potentially unstable
3. **Frequency Response Types**: Low-pass, high-pass, band-pass, band-stop

### Time-Frequency Analysis

**Limitation of Fourier Transform**: Provides frequency content but no temporal localization.

**Definition 1.6** (Short-Time Fourier Transform): 
$$X(n, \omega) = \sum_{m=-\infty}^{\infty} x[m]w[m-n]e^{-j\omega m}$$
where $w[n]$ is a window function.

**Gabor Transform**: STFT with Gaussian window, optimal time-frequency localization.

**Uncertainty Principle**: For any signal, time-frequency localization is bounded:
$$\sigma_t^2 \sigma_\omega^2 \geq \frac{1}{4}$$

### Wavelet Analysis

**Definition 1.7** (Continuous Wavelet Transform):
$$W(a, b) = \frac{1}{\sqrt{a}} \int_{-\infty}^{\infty} x(t) \psi^*\left(\frac{t-b}{a}\right) dt$$
where $\psi(t)$ is the mother wavelet, $a$ is scale, $b$ is translation.

**Discrete Wavelet Transform**: Dyadic scales $a = 2^j$, discrete translations $b = k2^j$.

**Multi-Resolution Analysis**: Decomposition into approximation and detail coefficients:
$$x(t) = \sum_k c_{J,k} \phi_{J,k}(t) + \sum_{j \leq J} \sum_k d_{j,k} \psi_{j,k}(t)$$

### Applications in Machine Learning and AI

**Computer Vision**:
1. **Convolutional Neural Networks**: Convolution as feature extraction
2. **Image Processing**: Filtering, edge detection, enhancement
3. **Feature Descriptors**: SIFT, HOG based on gradient computations
4. **Multi-scale Analysis**: Pyramid representations, scale-space theory

**Audio and Speech Processing**:
1. **Mel-Frequency Cepstral Coefficients**: Perceptually motivated features
2. **Spectrograms**: Time-frequency representations for deep learning
3. **Voice Activity Detection**: Energy-based and spectral methods
4. **Speech Enhancement**: Wiener filtering, spectral subtraction

**Time Series Analysis**:
1. **Trend Decomposition**: Seasonal, trend, and irregular components
2. **Change Point Detection**: Statistical tests on filtered signals
3. **Forecasting**: Spectral methods and frequency domain modeling
4. **Anomaly Detection**: Deviations from expected spectral patterns

**Natural Language Processing**:
1. **Signal Representations**: Embeddings as high-dimensional signals
2. **Attention Mechanisms**: Similarity as correlation functions
3. **Sequence Processing**: Convolutions and recurrent structures
4. **Transformer Architecture**: Positional encoding and frequency analysis

### Statistical Signal Processing

**Definition 1.8** (Random Process): Collection of random variables $\{X(t, \omega)\}$ indexed by time $t$ and outcome $\omega$.

**Stationarity**: 
- **Strict-Sense Stationary**: All finite-dimensional distributions invariant under time shifts
- **Wide-Sense Stationary**: Mean and autocorrelation depend only on time difference

**Power Spectral Density**: For WSS process with autocorrelation $R_x(\tau)$:
$$S_x(\omega) = \int_{-\infty}^{\infty} R_x(\tau) e^{-j\omega\tau} d\tau$$

**Wiener-Khintchine Theorem**: $S_x(\omega) = \lim_{T \to \infty} \mathbb{E}[|X_T(\omega)|^2]/T$

### Compressed Sensing and Sparse Signals

**Definition 1.9** (Sparse Signal): Signal $x \in \mathbb{R}^N$ is $k$-sparse if $\|x\|_0 \leq k \ll N$.

**Compressed Sensing Problem**: Recover $x$ from underdetermined measurements $y = \Phi x$ where $\Phi \in \mathbb{R}^{M \times N}, M \ll N$.

**Restricted Isometry Property**: Matrix $\Phi$ satisfies $(1-\delta_k)\|x\|_2^2 \leq \|\Phi x\|_2^2 \leq (1+\delta_k)\|x\|_2^2$ for all $k$-sparse $x$.

**Recovery Guarantees**: If $\Phi$ satisfies RIP with $\delta_{2k} < \sqrt{2} - 1$, then $\ell_1$ minimization recovers any $k$-sparse signal.

**Applications**:
- **Magnetic Resonance Imaging**: Faster acquisition with fewer measurements
- **Radar/Sonar**: High-resolution imaging with sparse arrays
- **Network Monitoring**: Identifying anomalies in sparse traffic patterns
- **Machine Learning**: Feature selection and dimensionality reduction

### Modern Signal Processing Paradigms

**Deep Learning Integration**:
1. **Learnable Transforms**: Trainable filter banks and wavelets
2. **End-to-End Processing**: Direct optimization of signal processing pipelines
3. **Physics-Informed Networks**: Incorporating signal processing constraints
4. **Differentiable DSP**: Gradient-based optimization of signal processing algorithms

**Quantum Signal Processing**:
1. **Quantum Fourier Transform**: Exponential speedup for certain problems
2. **Quantum Sensing**: Enhanced precision through quantum entanglement
3. **Quantum Machine Learning**: Signal processing on quantum computers

**Graph Signal Processing**:
1. **Signals on Graphs**: Extending classical concepts to irregular domains
2. **Graph Fourier Transform**: Eigendecomposition of graph Laplacian
3. **Graph Filtering**: Spectral and spatial domain approaches
4. **Applications**: Social networks, sensor networks, brain connectivity

### Advanced Mathematical Connections

**Harmonic Analysis**: Deep connections to abstract harmonic analysis:
- **Group Theory**: Translation and modulation as group actions
- **Representation Theory**: Irreducible representations and multiplicity
- **Abstract Wavelets**: Coherent states and admissible wavelets

**Functional Analysis**: Signal spaces as function spaces:
- **Banach Spaces**: $L^p$ spaces of signals
- **Hilbert Spaces**: Inner product structure and orthogonality
- **Operators**: Convolution, differentiation, and integral transforms

**Differential Geometry**: Signal processing on manifolds:
- **Manifold-Valued Signals**: Signals taking values in curved spaces
- **Geometric Deep Learning**: Neural networks on non-Euclidean domains
- **Diffusion on Manifolds**: Heat kernels and spectral methods

In [ ]:
## 2. Fourier Analysis and Harmonic Theory {#fourier-analysis}

Fourier analysis forms the theoretical backbone of signal processing, providing the mathematical framework for decomposing signals into their harmonic components. This section develops the rigorous mathematical foundations essential for understanding frequency domain analysis in machine learning and AI applications.

### Classical Fourier Series Theory

**Definition 2.1** (Fourier Series): For periodic function $f(t)$ with period $T$, the Fourier series representation is:
$$f(t) = \frac{a_0}{2} + \sum_{n=1}^{\infty} \left[a_n \cos\left(\frac{2\pi nt}{T}\right) + b_n \sin\left(\frac{2\pi nt}{T}\right)\right]$$

**Coefficients**:
$$a_n = \frac{2}{T} \int_{-T/2}^{T/2} f(t) \cos\left(\frac{2\pi nt}{T}\right) dt$$
$$b_n = \frac{2}{T} \int_{-T/2}^{T/2} f(t) \sin\left(\frac{2\pi nt}{T}\right) dt$$

**Complex Exponential Form**:
$$f(t) = \sum_{n=-\infty}^{\infty} c_n e^{j2\pi nt/T}$$
where $c_n = \frac{1}{T} \int_{-T/2}^{T/2} f(t) e^{-j2\pi nt/T} dt$

**Theorem 2.1** (Dirichlet Conditions): A periodic function $f(t)$ has a convergent Fourier series if:
1. $f(t)$ has finite number of discontinuities in one period
2. $f(t)$ has finite number of maxima and minima in one period  
3. $\int_{-T/2}^{T/2} |f(t)| dt < \infty$

### Fourier Transform Theory and Properties

**Definition 2.2** (Fourier Transform Pairs):
- **Analysis**: $X(\omega) = \int_{-\infty}^{\infty} x(t) e^{-j\omega t} dt$
- **Synthesis**: $x(t) = \frac{1}{2\pi} \int_{-\infty}^{\infty} X(\omega) e^{j\omega t} d\omega$

**Existence Conditions**: Fourier transform exists if:
1. $x(t) \in L^1(\mathbb{R})$: $\int_{-\infty}^{\infty} |x(t)| dt < \infty$
2. $x(t)$ has finite number of discontinuities
3. $x(t)$ has finite variation in any finite interval

**Fundamental Transform Properties**:

1. **Duality**: $\mathcal{F}\{\mathcal{F}\{x(t)\}\} = 2\pi x(-\omega)$
2. **Scaling**: $\mathcal{F}\{x(at)\} = \frac{1}{|a|} X\left(\frac{\omega}{a}\right)$
3. **Time Reversal**: $\mathcal{F}\{x(-t)\} = X(-\omega)$
4. **Differentiation**: $\mathcal{F}\left\{\frac{d^n x(t)}{dt^n}\right\} = (j\omega)^n X(\omega)$
5. **Integration**: $\mathcal{F}\left\{\int_{-\infty}^t x(\tau) d\tau\right\} = \frac{X(\omega)}{j\omega} + \pi X(0) \delta(\omega)$

**Theorem 2.2** (Parseval's Identity): For signals in $L^2(\mathbb{R})$:
$$\int_{-\infty}^{\infty} |x(t)|^2 dt = \frac{1}{2\pi} \int_{-\infty}^{\infty} |X(\omega)|^2 d\omega$$

**Physical Interpretation**: Energy is preserved between time and frequency domains.

### Generalized Functions and Distributions

**Definition 2.3** (Dirac Delta Function): The impulse function $\delta(t)$ is characterized by:
$$\int_{-\infty}^{\infty} \delta(t) f(t) dt = f(0)$$
for any continuous function $f(t)$.

**Properties**:
- $\delta(at) = \frac{1}{|a|} \delta(t)$
- $\mathcal{F}\{\delta(t)\} = 1$
- $\mathcal{F}\{1\} = 2\pi \delta(\omega)$

**Unit Step Function**: $u(t) = \begin{cases} 1 & t \geq 0 \\ 0 & t < 0 \end{cases}$

**Fourier Transform**: $\mathcal{F}\{u(t)\} = \frac{1}{j\omega} + \pi \delta(\omega)$

### Window Functions and Spectral Leakage

**Definition 2.4** (Window Function): A function $w(t)$ used to truncate infinite-duration signals for analysis.

**Common Windows**:
1. **Rectangular**: $w_R(t) = \begin{cases} 1 & |t| \leq T/2 \\ 0 & |t| > T/2 \end{cases}$
2. **Hamming**: $w_H(t) = 0.54 + 0.46 \cos(2\pi t/T)$
3. **Hanning**: $w_{Ha}(t) = 0.5 + 0.5 \cos(2\pi t/T)$
4. **Kaiser**: $w_K(t) = \frac{I_0(\beta\sqrt{1-(2t/T)^2})}{I_0(\beta)}$ where $I_0$ is modified Bessel function

**Windowing Effects**:
- **Main Lobe Width**: Frequency resolution
- **Side Lobe Level**: Spectral leakage characteristics
- **Scalloping Loss**: Amplitude reduction for off-bin frequencies

**Theorem 2.3** (Uncertainty Principle): For window function $w(t)$ with duration $\Delta t$ and bandwidth $\Delta \omega$:
$$\Delta t \cdot \Delta \omega \geq \frac{1}{2}$$

### Discrete-Time Fourier Analysis

**Definition 2.5** (Discrete-Time Fourier Transform): For sequence $x[n]$:
$$X(e^{j\omega}) = \sum_{n=-\infty}^{\infty} x[n] e^{-j\omega n}$$

**Inverse DTFT**:
$$x[n] = \frac{1}{2\pi} \int_{-\pi}^{\pi} X(e^{j\omega}) e^{j\omega n} d\omega$$

**Periodicity**: $X(e^{j(\omega + 2\pi)}) = X(e^{j\omega})$

**Definition 2.6** (Discrete Fourier Transform): For finite sequence $x[n], n = 0, \ldots, N-1$:
$$X[k] = \sum_{n=0}^{N-1} x[n] e^{-j2\pi kn/N}, \quad k = 0, \ldots, N-1$$

**Matrix Representation**: $\mathbf{X} = \mathbf{W}_N \mathbf{x}$ where $[\mathbf{W}_N]_{k,n} = e^{-j2\pi kn/N}$

**Properties of DFT Matrix**:
- **Unitary**: $\mathbf{W}_N^H \mathbf{W}_N = N \mathbf{I}$
- **Symmetric**: $[\mathbf{W}_N]_{k,n} = [\mathbf{W}_N]_{n,k}$
- **Circulant Structure**: Each row is cyclic shift of previous

### Fast Fourier Transform (FFT)

**Cooley-Tukey Algorithm**: Exploits periodicity and symmetry of DFT.

**Decimation-in-Time**: For $N = 2^m$:
$$X[k] = \sum_{n=0}^{N/2-1} x[2n] e^{-j2\pi k(2n)/N} + e^{-j2\pi k/N} \sum_{n=0}^{N/2-1} x[2n+1] e^{-j2\pi k(2n)/N}$$

**Computational Complexity**: $O(N \log N)$ vs $O(N^2)$ for direct DFT.

**Butterfly Operations**: Basic building blocks combining two complex numbers.

**Theorem 2.4** (FFT Efficiency): For length-$N$ DFT with $N = 2^m$:
- **Multiplications**: $\frac{N}{2} \log_2 N$
- **Additions**: $N \log_2 N$

### Advanced Fourier Methods

**Chirp-Z Transform**: Evaluates DFT on spiral contours in z-plane.

**Prime Factor Algorithm**: FFT for composite lengths using Chinese Remainder Theorem.

**Split-Radix FFT**: Optimal multiplication count for real-valued transforms.

**Fractional Fourier Transform**: Rotation in time-frequency plane by angle $\alpha$:
$$X_\alpha(u) = \int_{-\infty}^{\infty} x(t) K_\alpha(t,u) dt$$
where kernel $K_\alpha(t,u)$ involves Fresnel integrals.

### Applications in Machine Learning

**Convolutional Neural Networks**:
- **Frequency Domain Convolution**: $\mathcal{F}\{f * g\} = \mathcal{F}\{f\} \cdot \mathcal{F}\{g\}$
- **FFT-based Convolution**: Efficient for large kernels
- **Spectral CNN**: Learning in frequency domain

**Feature Extraction**:
- **Power Spectral Density**: $|X(\omega)|^2$ for frequency content analysis
- **Spectral Centroid**: $\frac{\sum \omega |X(\omega)|^2}{\sum |X(\omega)|^2}$
- **Spectral Rolloff**: Frequency below which 85% of energy concentrated

**Time Series Analysis**:
- **Spectral Estimation**: Periodogram, Welch's method, multitaper
- **Cross-Spectral Analysis**: Coherence and phase relationships
- **Frequency Domain Bootstrap**: Statistical inference in spectral domain

## 3. Discrete-Time Signal Processing {#discrete-time}

Discrete-time signal processing forms the computational foundation for digital systems, providing the mathematical framework for analyzing and processing sampled signals. This section develops rigorous theoretical foundations essential for understanding digital signal processing in machine learning and AI applications.

### Sampling and Reconstruction Theory

**Definition 3.1** (Sampling Process): The transformation from continuous-time signal $x_c(t)$ to discrete-time sequence:
$$x[n] = x_c(nT_s)$$
where $T_s$ is the sampling period and $f_s = 1/T_s$ is the sampling frequency.

**Mathematical Model of Sampling**:
$$x_s(t) = x_c(t) \sum_{n=-\infty}^{\infty} \delta(t - nT_s) = \sum_{n=-\infty}^{\infty} x_c(nT_s) \delta(t - nT_s)$$

**Frequency Domain Analysis**:
$$X_s(\omega) = \frac{1}{T_s} \sum_{k=-\infty}^{\infty} X_c\left(\omega - k\frac{2\pi}{T_s}\right)$$

**Theorem 3.1** (Sampling Theorem - Rigorous Statement): Let $x_c(t)$ be band-limited with $X_c(\omega) = 0$ for $|\omega| > \omega_c$. Then $x_c(t)$ can be exactly reconstructed from samples $x[n] = x_c(nT_s)$ if and only if:
$$\omega_s = \frac{2\pi}{T_s} > 2\omega_c$$

**Reconstruction Formula** (Whittaker-Shannon):
$$x_c(t) = \sum_{n=-\infty}^{\infty} x[n] \frac{\sin(\pi(t - nT_s)/T_s)}{\pi(t - nT_s)/T_s}$$

**Aliasing**: When $\omega_s \leq 2\omega_c$, spectral replicas overlap, causing:
$$X_s(\omega) \neq \frac{1}{T_s} X_c(\omega) \text{ for } |\omega| \leq \pi/T_s$$

### Z-Transform Theory

**Definition 3.2** (Z-Transform): For sequence $x[n]$:
$$X(z) = \mathcal{Z}\{x[n]\} = \sum_{n=-\infty}^{\infty} x[n] z^{-n}$$
where $z \in \mathbb{C}$ is a complex variable.

**Region of Convergence (ROC)**: Set of values $z$ for which $X(z)$ converges.

**Convergence Condition**: $\sum_{n=-\infty}^{\infty} |x[n]| |z|^{-n} < \infty$

**Properties of ROC**:
1. ROC is an annular region in z-plane
2. ROC cannot contain poles
3. For finite-duration sequences, ROC is entire z-plane except possibly $z = 0$ or $z = \infty$
4. For causal sequences, ROC extends outward from outermost pole
5. For anti-causal sequences, ROC extends inward from innermost pole

**Important Z-Transform Pairs**:
- $\delta[n] \leftrightarrow 1$, ROC: entire z-plane
- $u[n] \leftrightarrow \frac{1}{1-z^{-1}}$, ROC: $|z| > 1$
- $a^n u[n] \leftrightarrow \frac{1}{1-az^{-1}}$, ROC: $|z| > |a|$
- $n a^n u[n] \leftrightarrow \frac{az^{-1}}{(1-az^{-1})^2}$, ROC: $|z| > |a|$

**Fundamental Properties**:
1. **Linearity**: $\mathcal{Z}\{ax[n] + by[n]\} = aX(z) + bY(z)$
2. **Time Shift**: $\mathcal{Z}\{x[n-k]\} = z^{-k}X(z)$
3. **Time Reversal**: $\mathcal{Z}\{x[-n]\} = X(z^{-1})$
4. **Differentiation**: $\mathcal{Z}\{nx[n]\} = -z\frac{dX(z)}{dz}$
5. **Convolution**: $\mathcal{Z}\{x[n] * y[n]\} = X(z)Y(z)$

### Linear Time-Invariant Systems

**Definition 3.3** (LTI System Properties):
- **Linearity**: $T\{ax_1[n] + bx_2[n]\} = aT\{x_1[n]\} + bT\{x_2[n]\}$
- **Time-Invariance**: $T\{x[n-k]\} = y[n-k]$ where $y[n] = T\{x[n]\}$

**Impulse Response Characterization**: For LTI system with impulse response $h[n]$:
$$y[n] = \sum_{k=-\infty}^{\infty} x[k] h[n-k] = x[n] * h[n]$$

**System Function**: $H(z) = \mathcal{Z}\{h[n]\}$, giving $Y(z) = X(z)H(z)$

**Frequency Response**: $H(e^{j\omega}) = H(z)|_{z=e^{j\omega}}$

**System Classification**:
1. **Causal**: $h[n] = 0$ for $n < 0$
2. **Stable**: $\sum_{n=-\infty}^{\infty} |h[n]| < \infty$ (BIBO stability)
3. **Minimum Phase**: All zeros inside unit circle
4. **All-Pass**: $|H(e^{j\omega})| = 1$ for all $\omega$

**Theorem 3.2** (Stability Criterion): An LTI system is stable if and only if the ROC of $H(z)$ includes the unit circle.

### Difference Equations and System Realization

**Definition 3.4** (Linear Constant-Coefficient Difference Equation):
$$\sum_{k=0}^N a_k y[n-k] = \sum_{k=0}^M b_k x[n-k]$$

**System Function**:
$$H(z) = \frac{\sum_{k=0}^M b_k z^{-k}}{\sum_{k=0}^N a_k z^{-k}} = \frac{B(z)}{A(z)}$$

**Pole-Zero Analysis**:
- **Zeros**: Roots of $B(z) = 0$
- **Poles**: Roots of $A(z) = 0$
- **Stability**: All poles inside unit circle for causal stable system

**System Realizations**:

1. **Direct Form I**: Direct implementation of difference equation
2. **Direct Form II**: Canonical form with minimum delay elements
3. **Cascade Form**: Product of second-order sections
4. **Parallel Form**: Sum of first and second-order sections

**State-Space Representation**:
$$\mathbf{v}[n+1] = \mathbf{A}\mathbf{v}[n] + \mathbf{b}x[n]$$
$$y[n] = \mathbf{c}^T\mathbf{v}[n] + dx[n]$$

### Discrete Fourier Transform and FFT

**Definition 3.5** (DFT Properties):
For $N$-point DFT: $X[k] = \sum_{n=0}^{N-1} x[n] e^{-j2\pi kn/N}$

**Fundamental Properties**:
1. **Periodicity**: $X[k+N] = X[k]$, $x[n+N] = x[n]$
2. **Circular Shift**: $x[(n-m) \bmod N] \leftrightarrow X[k]e^{-j2\pi km/N}$
3. **Circular Convolution**: $x[n] \circledast y[n] \leftrightarrow X[k]Y[k]$
4. **Parseval's Theorem**: $\sum_{n=0}^{N-1} |x[n]|^2 = \frac{1}{N} \sum_{k=0}^{N-1} |X[k]|^2$

**Linear vs Circular Convolution**: For sequences of length $L$ and $M$:
- Linear convolution length: $L + M - 1$
- Circular convolution length: $N = \max(L, M)$
- To perform linear convolution via DFT: zero-pad to length $L + M - 1$

**FFT Implementation Details**:

**Radix-2 Decimation-in-Time**:
$$X[k] = \sum_{n=0}^{N/2-1} x[2n] W_N^{2nk} + W_N^k \sum_{n=0}^{N/2-1} x[2n+1] W_N^{2nk}$$
where $W_N = e^{-j2\pi/N}$

**Computational Complexity**:
- **Direct DFT**: $O(N^2)$ complex multiplications
- **Radix-2 FFT**: $O(N \log_2 N)$ complex multiplications
- **Speedup Factor**: $N/(2\log_2 N)$ for large $N$

**Bit-Reversal**: Input reordering required for in-place computation.

### Advanced DFT Techniques

**Overlap-Add Method**: For long sequence convolution:
1. Partition input into blocks of length $L$
2. Zero-pad each block to length $L + M - 1$
3. Convolve each block with filter via FFT
4. Overlap and add results

**Overlap-Save Method**: Alternative for block convolution:
1. Overlap input blocks by $M - 1$ samples
2. Perform circular convolution of length $L$
3. Save last $L - M + 1$ samples from each block

**Chirp-Z Transform (CZT)**: Evaluate DFT at arbitrary points:
$$X(z_k) = \sum_{n=0}^{N-1} x[n] z_k^{-n}$$
where $z_k = AW^{-k}$ for complex constants $A, W$.

### Quantization and Finite Word Length Effects

**Definition 3.6** (Quantization): Mapping continuous amplitude values to finite set of discrete levels.

**Uniform Quantization**: For $B$-bit quantizer with full-scale range $\pm X_m$:
$$\Delta = \frac{2X_m}{2^B}$$

**Quantization Noise Model**:
- **Signal-to-Quantization-Noise Ratio**: $SQNR = 6.02B + 1.76$ dB for uniform quantization
- **Quantization Error**: $e[n] = x_Q[n] - x[n]$, modeled as uniform white noise

**Fixed-Point Arithmetic**:
- **Overflow**: When result exceeds representable range
- **Truncation vs Rounding**: Different quantization strategies
- **Limit Cycles**: Oscillations due to nonlinear quantization

**Coefficient Quantization**: Effects on pole/zero locations and frequency response.

### Multirate Signal Processing Fundamentals

**Definition 3.7** (Decimation): Reduce sampling rate by factor $M$:
$$y[n] = x[nM]$$

**Frequency Domain**: $Y(e^{j\omega}) = \frac{1}{M} \sum_{k=0}^{M-1} X(e^{j(\omega-2\pi k)/M})$

**Definition 3.8** (Interpolation): Increase sampling rate by factor $L$:
$$y[n] = \begin{cases} x[n/L] & n = 0, \pm L, \pm 2L, \ldots \\ 0 & \text{otherwise} \end{cases}$$

**Frequency Domain**: $Y(e^{j\omega}) = X(e^{j\omega L})$

**Polyphase Decomposition**: For filter $h[n]$ and decimation factor $M$:
$$h[n] = \sum_{k=0}^{M-1} e_k[n/M] \delta[n - k \bmod M]$$
where $e_k[n] = h[nM + k]$ are polyphase components.

**Noble Identities**: Commutation of sampling rate converters with filters.

### Applications in Machine Learning

**Audio Processing**:
- **Spectral Features**: MFCCs, spectral centroid, zero-crossing rate
- **Short-Time Fourier Transform**: Time-frequency analysis for speech
- **Pitch Detection**: Autocorrelation and cepstral methods

**Computer Vision**:
- **Image Filtering**: 2D convolution for edge detection, smoothing
- **Pyramid Representations**: Multiresolution analysis
- **Optical Flow**: Motion estimation using temporal filtering

**Time Series Analysis**:
- **Digital Filtering**: Noise reduction and trend extraction
- **Spectral Estimation**: Power spectral density for frequency analysis
- **Change Point Detection**: Statistical tests on filtered signals

**Signal Enhancement**:
- **Wiener Filtering**: Optimal linear filtering for noise reduction
- **Adaptive Filtering**: LMS and RLS algorithms
- **Blind Source Separation**: ICA and matrix factorization techniques

## 4. Digital Filter Design and Analysis {#digital-filtering}

Digital filter design combines mathematical theory with practical implementation constraints to create systems that modify signals in desired ways. This section develops the theoretical foundations essential for understanding filter design in machine learning and signal processing applications.

### Filter Specifications and Performance Metrics

**Definition 4.1** (Filter Specifications): A digital filter is characterized by:
- **Passband**: Frequency range where signal passes with minimal attenuation
- **Stopband**: Frequency range where signal is significantly attenuated  
- **Transition Band**: Frequency range between passband and stopband
- **Ripple**: Acceptable variation in passband/stopband response

**Performance Metrics**:
1. **Passband Ripple**: $\delta_p = 10^{A_p/20} - 1$ where $A_p$ is passband attenuation in dB
2. **Stopband Attenuation**: $A_s$ in dB, typically $> 40$ dB
3. **Transition Width**: $\Delta f = f_s - f_p$ (normalized by sampling frequency)
4. **Group Delay**: $\tau(\omega) = -\frac{d\phi(\omega)}{d\omega}$ where $\phi(\omega)$ is phase response

### FIR Filter Design Methods

**Definition 4.2** (FIR Filter): Finite Impulse Response filter with system function:
$$H(z) = \sum_{n=0}^{N-1} h[n] z^{-n}$$

**Advantages**: Always stable, linear phase possible, finite precision effects minimal.

**Window Method**: Design ideal filter and truncate with window function.

**Ideal Low-Pass Filter**: 
$$h_d[n] = \frac{\omega_c}{\pi} \text{sinc}\left(\frac{\omega_c(n - \alpha)}{\pi}\right)$$
where $\alpha = (N-1)/2$ for linear phase and $\omega_c$ is cutoff frequency.

**Windowed Filter**: $h[n] = h_d[n] \cdot w[n]$ where $w[n]$ is window function.

**Trade-offs**:
- **Rectangular Window**: Sharpest transition, highest ripple
- **Hamming Window**: Good ripple suppression, wider transition
- **Kaiser Window**: Adjustable parameter $\beta$ controls trade-off

**Frequency Sampling Method**: Specify desired frequency response at DFT frequencies:
$$H[k] = H_d(e^{j2\pi k/N}), \quad k = 0, 1, \ldots, N-1$$
$$h[n] = \frac{1}{N} \sum_{k=0}^{N-1} H[k] e^{j2\pi kn/N}$$

**Parks-McClellan Algorithm** (Optimal Equiripple Design):

**Theorem 4.1** (Remez Exchange): The optimal FIR filter minimizing maximum approximation error has equiripple behavior with $(N+2)/2$ extrema for Type I filters.

**Algorithm**:
1. Initialize extremal frequencies
2. Solve for filter coefficients using interpolation
3. Find new extremal frequencies
4. Iterate until convergence

**Linear Phase Constraints**: For real-valued filters with linear phase:
- **Type I**: $N$ odd, symmetric about center, arbitrary frequency response
- **Type II**: $N$ even, symmetric, zero response at Nyquist frequency
- **Type III**: $N$ odd, antisymmetric, zero response at DC and Nyquist
- **Type IV**: $N$ even, antisymmetric, zero response at DC

### IIR Filter Design Methods

**Definition 4.3** (IIR Filter): Infinite Impulse Response filter with system function:
$$H(z) = \frac{\sum_{k=0}^M b_k z^{-k}}{\sum_{k=0}^N a_k z^{-k}}$$

**Advantages**: Lower order for given specifications, efficient implementation.
**Disadvantages**: Potential instability, nonlinear phase, quantization sensitivity.

**Analog Filter Prototypes**:

1. **Butterworth Filter**: Maximally flat passband response
   $$|H_a(j\Omega)|^2 = \frac{1}{1 + (\Omega/\Omega_c)^{2N}}$$

2. **Chebyshev Type I**: Equiripple in passband, monotonic in stopband
   $$|H_a(j\Omega)|^2 = \frac{1}{1 + \epsilon^2 T_N^2(\Omega/\Omega_c)}$$
   where $T_N$ is Chebyshev polynomial of first kind.

3. **Chebyshev Type II**: Monotonic in passband, equiripple in stopband

4. **Elliptic (Cauer)**: Equiripple in both passband and stopband
   $$|H_a(j\Omega)|^2 = \frac{1}{1 + \epsilon^2 U_N^2(\Omega/\Omega_c)}$$
   where $U_N$ is Jacobian elliptic function.

**Analog-to-Digital Transformation Methods**:

**Impulse Invariance**: Match impulse responses
$$h[n] = T_s h_a(nT_s)$$
**Limitation**: Aliasing for non-band-limited filters.

**Bilinear Transform**: Maps s-plane to z-plane
$$s = \frac{2}{T_s} \frac{1 - z^{-1}}{1 + z^{-1}}$$
**Properties**: 
- Maps entire jΩ axis to unit circle
- No aliasing but frequency warping occurs
- **Prewarping**: $\Omega_a = \frac{2}{T_s} \tan(\omega_d T_s/2)$

### Advanced Filter Structures

**Direct Form Realizations**:
- **Direct Form I**: Direct implementation of difference equation
- **Direct Form II**: Canonical form with $\max(M,N)$ delay elements

**Cascade Realization**: Factorize $H(z)$ into second-order sections:
$$H(z) = \prod_{k=1}^{K} H_k(z) = \prod_{k=1}^{K} \frac{b_{k0} + b_{k1}z^{-1} + b_{k2}z^{-2}}{1 + a_{k1}z^{-1} + a_{k2}z^{-2}}$$

**Parallel Realization**: Partial fraction expansion:
$$H(z) = H_0(z) + \sum_{k=1}^{K} H_k(z)$$

**Lattice Structures**: Based on reflection coefficients, naturally stable.

**State-Space Realization**: Matrix representation
$$\mathbf{v}[n+1] = \mathbf{A}\mathbf{v}[n] + \mathbf{b}x[n]$$
$$y[n] = \mathbf{c}^T\mathbf{v}[n] + dx[n]$$

### Multirate Filter Design

**Definition 4.4** (Polyphase Decomposition): For filter $h[n]$ and factor $M$:
$$h[n] = \sum_{k=0}^{M-1} e_k[n/M] \delta[(n-k) \bmod M]$$
where $e_k[m] = h[mM + k]$ are polyphase components.

**Polyphase Matrix**: $\mathbf{E}(z) = [E_0(z), E_1(z), \ldots, E_{M-1}(z)]^T$

**Efficient Decimation**: 
$$Y(z) = \frac{1}{M} \sum_{k=0}^{M-1} \mathbf{e}^T(z^{1/M} W_M^k) X(z^{1/M} W_M^k)$$

**Efficient Interpolation**:
$$Y(z) = \mathbf{e}^T(z^L) X(z^L)$$

**Perfect Reconstruction Filter Banks**: Analysis and synthesis filters satisfy:
$$\sum_{k=0}^{M-1} H_k(z) G_k(z) = cz^{-n_0}$$

### Adaptive Filter Theory

**Definition 4.5** (Adaptive Filter): Filter with time-varying coefficients that adapt to optimize performance criterion.

**Least Mean Squares (LMS) Algorithm**:
$$\mathbf{w}[n+1] = \mathbf{w}[n] + \mu e[n] \mathbf{x}[n]$$
where $e[n] = d[n] - \mathbf{w}^T[n]\mathbf{x}[n]$ is error signal.

**Convergence Properties**:
- **Stability**: $0 < \mu < \frac{2}{\lambda_{\max}}$ where $\lambda_{\max}$ is largest eigenvalue of $\mathbf{R}_{xx}$
- **Steady-State MSE**: $\xi_{\min} + \frac{\mu}{2-\mu} \text{tr}(\mathbf{R}_{xx}) \xi_{\min}$

**Recursive Least Squares (RLS)**:
$$\mathbf{P}[n] = \frac{1}{\lambda}(\mathbf{P}[n-1] - \frac{\mathbf{P}[n-1]\mathbf{x}[n]\mathbf{x}^T[n]\mathbf{P}[n-1]}{\lambda + \mathbf{x}^T[n]\mathbf{P}[n-1]\mathbf{x}[n]})$$
$$\mathbf{w}[n] = \mathbf{w}[n-1] + \mathbf{P}[n]\mathbf{x}[n]e[n]$$

**Fast Convergence**: RLS converges in approximately $2M$ iterations vs $\sim 10M$ for LMS.

### Statistical Signal Processing Applications

**Wiener Filtering**: Optimal linear filter for known statistics
$$\mathbf{w}_{\text{opt}} = \mathbf{R}_{xx}^{-1} \mathbf{r}_{dx}$$
**Minimum MSE**: $\xi_{\min} = \sigma_d^2 - \mathbf{r}_{dx}^T \mathbf{R}_{xx}^{-1} \mathbf{r}_{dx}$

**Kalman Filtering**: Optimal state estimation for linear dynamic systems
$$\hat{\mathbf{x}}[n|n] = \hat{\mathbf{x}}[n|n-1] + \mathbf{K}[n](\mathbf{y}[n] - \mathbf{C}\hat{\mathbf{x}}[n|n-1])$$
where $\mathbf{K}[n]$ is Kalman gain matrix.

**Spectral Estimation**:
- **Periodogram**: $\hat{S}_{xx}(\omega) = \frac{1}{N}|X(\omega)|^2$
- **Welch Method**: Averaged periodograms with overlapping segments
- **Parametric Methods**: AR, MA, ARMA modeling

### Machine Learning Applications

**Convolutional Neural Networks**:
- **Learnable Filters**: CNNs as adaptive FIR filters
- **Backpropagation**: Gradient-based filter optimization
- **Receptive Fields**: Effective filter support regions

**Signal Preprocessing**:
- **Anti-aliasing**: Lowpass filtering before downsampling
- **Denoising**: Wiener filtering, wavelet denoising
- **Feature Extraction**: Bandpass filtering for specific frequency components

**Time Series Processing**:
- **Trend Removal**: High-pass filtering
- **Seasonal Decomposition**: Band-stop filtering
- **Anomaly Detection**: Matched filtering for known patterns

**Audio Processing**:
- **Equalizers**: Multi-band filtering systems
- **Noise Cancellation**: Adaptive filtering algorithms
- **Compression**: Perceptual audio coding with filter banks

### Implementation Considerations

**Finite Precision Effects**:
- **Coefficient Quantization**: Pole/zero location sensitivity
- **Round-off Noise**: Accumulation in recursive structures
- **Overflow**: Scaling strategies and saturation arithmetic

**Computational Complexity**:
- **FIR Filters**: $N$ multiplications per output sample
- **IIR Filters**: $M + N + 1$ multiplications per output sample
- **FFT-based Convolution**: $O(\log N)$ per sample for long filters

**Real-time Implementation**:
- **Latency**: Group delay and processing delay
- **Throughput**: Maximum sustainable sample rate
- **Memory Requirements**: Buffer sizes and coefficient storage

In [ ]:
# Advanced Signal Processing: Mathematical Framework and Implementation
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, linalg, optimize, sparse
from scipy.fft import fft, ifft, fftfreq, fftshift, rfft, irfft
from scipy.special import factorial, gamma, beta, bessel
from sklearn.decomposition import FastICA, PCA
from sklearn.linear_model import OrthogonalMatchingPursuit
import warnings
warnings.filterwarnings('ignore')

class AdvancedSignalProcessingFramework:
    """
    Comprehensive framework for advanced signal processing theory and applications.
    Covers mathematical foundations, practical algorithms, and ML applications.
    """
    
    def __init__(self):
        self.fs = 1000  # Default sampling frequency
        np.random.seed(42)  # For reproducible results
        
    def demonstrate_signal_spaces_and_transforms(self):
        """
        Mathematical foundations: Signal spaces, norms, and transform theory
        """
        print("=== SIGNAL SPACES AND MATHEMATICAL FOUNDATIONS ===")
        
        # Signal space examples
        N = 512
        t = np.linspace(0, 1, N, endpoint=False)
        
        # L^2 signal (energy signal)
        x_energy = np.exp(-5*t) * np.sin(2*np.pi*10*t)
        energy = np.trapz(x_energy**2, t)
        
        # L^1 signal (absolutely integrable)
        x_l1 = np.exp(-3*np.abs(t - 0.5))
        l1_norm = np.trapz(np.abs(x_l1), t)
        
        # Band-limited signal
        freqs = fftfreq(N, t[1] - t[0])
        X = fft(x_energy)
        bandwidth = np.max(freqs[np.abs(X) > 0.01*np.max(np.abs(X))])
        
        # Demonstrate Parseval's theorem
        time_energy = np.trapz(x_energy**2, t)
        freq_energy = np.sum(np.abs(X)**2) / N**2 * (t[-1] - t[0])
        parseval_error = np.abs(time_energy - freq_energy) / time_energy
        
        # Signal correlation and inner products
        x2 = np.sin(2*np.pi*15*t) * np.exp(-2*t)
        correlation = np.trapz(x_energy * x2, t)
        norm_x1 = np.sqrt(np.trapz(x_energy**2, t))
        norm_x2 = np.sqrt(np.trapz(x2**2, t))
        normalized_correlation = correlation / (norm_x1 * norm_x2)
        
        results = {
            'energy_signal': {'signal': x_energy, 'energy': energy},
            'l1_signal': {'signal': x_l1, 'l1_norm': l1_norm},
            'bandwidth': bandwidth,
            'parseval_error': parseval_error,
            'signal_correlation': normalized_correlation,
            'hilbert_space_properties': {
                'inner_product': correlation,
                'norm1': norm_x1,
                'norm2': norm_x2,
                'cauchy_schwarz_satisfied': abs(correlation) <= norm_x1 * norm_x2
            }
        }
        
        print(f"Energy of L^2 signal: {energy:.4f}")
        print(f"L^1 norm: {l1_norm:.4f}")
        print(f"Signal bandwidth: {bandwidth:.2f} Hz")
        print(f"Parseval's theorem error: {parseval_error:.2e}")
        print(f"Normalized correlation: {normalized_correlation:.4f}")
        
        return results
    
    def demonstrate_advanced_fourier_analysis(self):
        """
        Advanced Fourier analysis: Uncertainty principle, windowing, time-frequency
        """
        print("\n=== ADVANCED FOURIER ANALYSIS ===")
        
        N = 1024
        t = np.linspace(-5, 5, N)
        dt = t[1] - t[0]
        
        # Gaussian signal (optimal time-frequency localization)
        sigma_t = 0.5
        x_gaussian = np.exp(-t**2 / (2*sigma_t**2))
        
        # Fourier transform
        X_gaussian = fftshift(fft(fftshift(x_gaussian))) * dt
        freqs = fftshift(fftfreq(N, dt))
        
        # Time and frequency spreads
        t_mean = np.trapz(t * np.abs(x_gaussian)**2, t) / np.trapz(np.abs(x_gaussian)**2, t)
        f_mean = np.trapz(freqs * np.abs(X_gaussian)**2, freqs) / np.trapz(np.abs(X_gaussian)**2, freqs)
        
        sigma_t_measured = np.sqrt(np.trapz((t - t_mean)**2 * np.abs(x_gaussian)**2, t) / 
                                  np.trapz(np.abs(x_gaussian)**2, t))
        sigma_f_measured = np.sqrt(np.trapz((freqs - f_mean)**2 * np.abs(X_gaussian)**2, freqs) / 
                                  np.trapz(np.abs(X_gaussian)**2, freqs))
        
        # Uncertainty product
        uncertainty_product = sigma_t_measured * sigma_f_measured
        theoretical_minimum = 1 / (4 * np.pi)
        
        # Short-Time Fourier Transform
        window_size = 64
        hop_size = 16
        stft_result = self._compute_stft(x_gaussian, window_size, hop_size)
        
        # Gabor transform (STFT with Gaussian window)
        gabor_window = signal.gaussian(window_size, std=window_size/6)
        gabor_result = self._compute_stft(x_gaussian, window_size, hop_size, gabor_window)
        
        results = {
            'gaussian_signal': {'time': t, 'signal': x_gaussian, 'spectrum': (freqs, X_gaussian)},
            'uncertainty_principle': {
                'sigma_t': sigma_t_measured,
                'sigma_f': sigma_f_measured,
                'product': uncertainty_product,
                'theoretical_minimum': theoretical_minimum,
                'ratio': uncertainty_product / theoretical_minimum
            },
            'stft': stft_result,
            'gabor': gabor_result
        }
        
        print(f"Time spread (σt): {sigma_t_measured:.4f}")
        print(f"Frequency spread (σf): {sigma_f_measured:.4f}")
        print(f"Uncertainty product: {uncertainty_product:.4f}")
        print(f"Theoretical minimum: {theoretical_minimum:.4f}")
        print(f"Ratio to minimum: {uncertainty_product/theoretical_minimum:.4f}")
        
        return results
    
    def demonstrate_wavelet_analysis(self):
        """
        Wavelet analysis: Continuous and discrete wavelets, multiresolution analysis
        """
        print("\n=== WAVELET ANALYSIS ===")
        
        # Generate test signal with multiple scales
        N = 1024
        t = np.linspace(0, 1, N, endpoint=False)
        x = (np.sin(2*np.pi*10*t) * (t < 0.3) + 
             np.sin(2*np.pi*50*t) * ((t >= 0.3) & (t < 0.6)) + 
             np.sin(2*np.pi*100*t) * (t >= 0.6))
        x += 0.1 * np.random.randn(N)
        
        # Morlet wavelet (complex)
        def morlet_wavelet(t, w0=6):
            return np.exp(1j*w0*t) * np.exp(-t**2/2) * np.pi**(-0.25)
        
        # Continuous Wavelet Transform
        scales = np.logspace(0, 3, 50)
        cwt_coeffs = np.zeros((len(scales), N), dtype=complex)
        
        for i, scale in enumerate(scales):
            # Scaled and translated wavelets
            for n in range(N):
                # Translation parameter
                b = t[n]
                # Wavelet at current scale and translation
                psi = morlet_wavelet((t - b) / scale) / np.sqrt(scale)
                # Wavelet coefficient
                cwt_coeffs[i, n] = np.trapz(x * np.conj(psi), t)
        
        # Discrete Wavelet Transform using filter banks
        # Daubechies wavelets
        def daubechies_filters(N):
            """Generate Daubechies wavelet filters"""
            if N == 4:  # DB2
                h = np.array([0.4829629131, 0.8365163037, 0.2241438680, -0.1294095226])
                g = np.array([-0.1294095226, -0.2241438680, 0.8365163037, -0.4829629131])
            else:  # DB4
                h = np.array([0.2303778133, 0.7148465706, 0.6308807679, -0.0279837694,
                             -0.1870348117, 0.0308413818, 0.0328830117, -0.0105974018])
                g = h[::-1] * np.array([(-1)**k for k in range(len(h))])
            return h, g
        
        h_lpf, h_hpf = daubechies_filters(4)
        
        # Multi-level decomposition
        levels = 6
        dwt_coeffs = self._dwt_decomposition(x, h_lpf, h_hpf, levels)
        
        # Perfect reconstruction
        x_reconstructed = self._dwt_reconstruction(dwt_coeffs, h_lpf, h_hpf)
        reconstruction_error = np.mean((x - x_reconstructed[:len(x)])**2)
        
        # Wavelet denoising using soft thresholding
        sigma_noise = 0.1
        threshold = sigma_noise * np.sqrt(2 * np.log(len(x)))
        dwt_denoised = self._wavelet_denoise(dwt_coeffs, threshold)
        x_denoised = self._dwt_reconstruction(dwt_denoised, h_lpf, h_hpf)
        
        results = {
            'original_signal': {'time': t, 'signal': x},
            'cwt': {'scales': scales, 'coefficients': cwt_coeffs},
            'dwt': {'coefficients': dwt_coeffs, 'levels': levels},
            'reconstruction': {
                'reconstructed': x_reconstructed[:len(x)],
                'error': reconstruction_error
            },
            'denoising': {
                'denoised_coeffs': dwt_denoised,
                'denoised_signal': x_denoised[:len(x)],
                'threshold': threshold
            }
        }
        
        print(f"DWT reconstruction error: {reconstruction_error:.2e}")
        print(f"Wavelet denoising threshold: {threshold:.4f}")
        
        return results
    
    def demonstrate_digital_filter_design(self):
        """
        Advanced digital filter design: IIR, FIR, optimal filters
        """
        print("\n=== DIGITAL FILTER DESIGN ===")
        
        # Filter specifications
        fs = 1000
        passband_freq = 100  # Hz
        stopband_freq = 150  # Hz
        passband_ripple = 0.5  # dB
        stopband_attenuation = 60  # dB
        
        # FIR filter design using Parks-McClellan algorithm
        nyquist = fs / 2
        bands = [0, passband_freq, stopband_freq, nyquist]
        desired = [1, 0]
        weights = [1, 10]  # Higher weight on stopband
        
        # Calculate required filter order
        A = -20 * np.log10(10**(-stopband_attenuation/20))
        delta_f = (stopband_freq - passband_freq) / fs
        N_fir = int(np.ceil((A - 13) / (14.6 * delta_f))) + 1
        
        # Design optimal FIR filter
        fir_optimal = signal.remez(N_fir, bands, desired, weights, fs=fs)
        
        # IIR filter design using analog prototypes
        # Butterworth filter
        N_butter, Wn_butter = signal.buttord(passband_freq, stopband_freq, 
                                           passband_ripple, stopband_attenuation, fs=fs)
        sos_butter = signal.butter(N_butter, Wn_butter, btype='low', output='sos', fs=fs)
        
        # Chebyshev Type I filter
        N_cheby1, Wn_cheby1 = signal.cheb1ord(passband_freq, stopband_freq,
                                             passband_ripple, stopband_attenuation, fs=fs)
        sos_cheby1 = signal.cheby1(N_cheby1, passband_ripple, Wn_cheby1, 
                                  btype='low', output='sos', fs=fs)
        
        # Elliptic filter (optimal)
        N_ellip, Wn_ellip = signal.ellipord(passband_freq, stopband_freq,
                                           passband_ripple, stopband_attenuation, fs=fs)
        sos_ellip = signal.ellip(N_ellip, passband_ripple, stopband_attenuation, 
                                Wn_ellip, btype='low', output='sos', fs=fs)
        
        # Frequency responses
        w = np.logspace(-1, np.log10(nyquist), 1000)
        
        _, h_fir = signal.freqz(fir_optimal, 1, worN=w, fs=fs)
        _, h_butter = signal.sosfreqz(sos_butter, worN=w, fs=fs)
        _, h_cheby1 = signal.sosfreqz(sos_cheby1, worN=w, fs=fs)
        _, h_ellip = signal.sosfreqz(sos_ellip, worN=w, fs=fs)
        
        # Group delay analysis
        _, gd_fir = signal.group_delay((fir_optimal, 1), w=w, fs=fs)
        _, gd_butter = signal.group_delay(sos_butter, w=w, fs=fs)
        _, gd_ellip = signal.group_delay(sos_ellip, w=w, fs=fs)
        
        # Filter comparison metrics
        filters = {
            'FIR_Optimal': {'order': len(fir_optimal)-1, 'response': h_fir, 'group_delay': gd_fir},
            'Butterworth': {'order': N_butter, 'response': h_butter, 'group_delay': gd_butter},
            'Chebyshev_I': {'order': N_cheby1, 'response': h_cheby1, 'group_delay': None},
            'Elliptic': {'order': N_ellip, 'response': h_ellip, 'group_delay': gd_ellip}
        }
        
        results = {
            'specifications': {
                'passband_freq': passband_freq,
                'stopband_freq': stopband_freq,
                'passband_ripple': passband_ripple,
                'stopband_attenuation': stopband_attenuation
            },
            'frequency': w,
            'filters': filters,
            'coefficients': {
                'fir': fir_optimal,
                'butterworth': sos_butter,
                'chebyshev1': sos_cheby1,
                'elliptic': sos_ellip
            }
        }
        
        print(f"FIR filter order: {len(fir_optimal)-1}")
        print(f"Butterworth order: {N_butter}")
        print(f"Chebyshev I order: {N_cheby1}")
        print(f"Elliptic order: {N_ellip}")
        
        return results
    
    def demonstrate_adaptive_filtering(self):
        """
        Adaptive filtering: LMS, RLS, applications
        """
        print("\n=== ADAPTIVE FILTERING ===")
        
        # System identification scenario
        N = 1000
        M = 8  # Adaptive filter length
        
        # Unknown system (to be identified)
        h_unknown = np.random.randn(M) * 0.1
        h_unknown[0] = 1.0  # Ensure non-trivial response
        
        # Input signal (white noise)
        x = np.random.randn(N)
        
        # Desired signal (output of unknown system + noise)
        d = np.convolve(x, h_unknown, mode='full')[:N]
        d += 0.05 * np.random.randn(N)  # Measurement noise
        
        # LMS Algorithm
        w_lms = np.zeros((N, M))
        e_lms = np.zeros(N)
        mu = 0.01  # Step size
        
        for n in range(M, N):
            x_n = x[n:n-M:-1] if n >= M else np.pad(x[n::-1], (0, M-n-1), 'constant')
            y_n = np.dot(w_lms[n-1], x_n)
            e_lms[n] = d[n] - y_n
            w_lms[n] = w_lms[n-1] + mu * e_lms[n] * x_n
        
        # RLS Algorithm
        w_rls = np.zeros((N, M))
        e_rls = np.zeros(N)
        P = np.eye(M) * 1000  # Inverse correlation matrix
        lam = 0.99  # Forgetting factor
        
        for n in range(M, N):
            x_n = x[n:n-M:-1] if n >= M else np.pad(x[n::-1], (0, M-n-1), 'constant')
            
            # Kalman gain
            k = P @ x_n / (lam + x_n.T @ P @ x_n)
            
            # A priori error
            e_rls[n] = d[n] - np.dot(w_rls[n-1], x_n)
            
            # Update weights
            w_rls[n] = w_rls[n-1] + k * e_rls[n]
            
            # Update inverse correlation matrix
            P = (P - np.outer(k, x_n.T @ P)) / lam
        
        # Performance analysis
        mse_lms = np.mean(e_lms**2)
        mse_rls = np.mean(e_rls**2)
        
        # Final weight estimates
        w_final_lms = w_lms[-1]
        w_final_rls = w_rls[-1]
        
        # Identification errors
        error_lms = np.linalg.norm(w_final_lms - h_unknown) / np.linalg.norm(h_unknown)
        error_rls = np.linalg.norm(w_final_rls - h_unknown) / np.linalg.norm(h_unknown)
        
        results = {
            'unknown_system': h_unknown,
            'lms': {
                'weights': w_lms,
                'error': e_lms,
                'final_weights': w_final_lms,
                'mse': mse_lms,
                'identification_error': error_lms,
                'step_size': mu
            },
            'rls': {
                'weights': w_rls,
                'error': e_rls,
                'final_weights': w_final_rls,
                'mse': mse_rls,
                'identification_error': error_rls,
                'forgetting_factor': lam
            },
            'input_signal': x,
            'desired_signal': d
        }
        
        print(f"LMS MSE: {mse_lms:.6f}")
        print(f"RLS MSE: {mse_rls:.6f}")
        print(f"LMS identification error: {error_lms:.4f}")
        print(f"RLS identification error: {error_rls:.4f}")
        
        return results
    
    def demonstrate_spectral_estimation(self):
        """
        Spectral estimation: Parametric and non-parametric methods
        """
        print("\n=== SPECTRAL ESTIMATION ===")
        
        # Generate AR process
        N = 512
        ar_coeffs = np.array([1, -1.5, 0.7])  # AR(2) process
        noise_var = 0.1
        
        # Generate AR signal
        x = np.random.randn(N)
        for n in range(2, N):
            x[n] = 1.5*x[n-1] - 0.7*x[n-2] + np.sqrt(noise_var)*np.random.randn()
        
        # Non-parametric methods
        
        # 1. Periodogram
        freqs_p, psd_periodogram = signal.periodogram(x, fs=1.0, window='hann')
        
        # 2. Welch's method
        freqs_w, psd_welch = signal.welch(x, fs=1.0, nperseg=128, noverlap=64)
        
        # 3. Multitaper method
        from scipy.signal.windows import dpss
        K = 4  # Number of tapers
        NW = 2.5  # Time-bandwidth product
        tapers = dpss(N, NW, K)
        
        psd_multitaper = np.zeros(N//2 + 1)
        for k in range(K):
            X_k = fft(x * tapers[k])
            psd_multitaper += np.abs(X_k[:N//2 + 1])**2
        psd_multitaper /= K
        freqs_mt = fftfreq(N, 1.0)[:N//2 + 1]
        
        # Parametric methods
        
        # 4. AR model estimation (Yule-Walker)
        ar_order = 10
        ar_coeffs_est, noise_var_est = self._ar_yule_walker(x, ar_order)
        
        # AR spectrum
        freqs_ar = np.linspace(0, 0.5, 256)
        H_ar = np.zeros(len(freqs_ar), dtype=complex)
        for i, f in enumerate(freqs_ar):
            z = np.exp(-1j * 2 * np.pi * f)
            H_ar[i] = 1 / (1 + np.sum(ar_coeffs_est[1:] * z**(-np.arange(1, ar_order+1))))
        
        psd_ar = noise_var_est * np.abs(H_ar)**2
        
        # 5. MUSIC algorithm (for sinusoids in noise)
        # Generate signal with sinusoids
        n = np.arange(N)
        f1, f2 = 0.1, 0.15  # Normalized frequencies
        x_sinusoids = (np.sin(2*np.pi*f1*n) + np.sin(2*np.pi*f2*n) + 
                      0.3*np.random.randn(N))
        
        # Autocorrelation matrix
        M = 20  # Model order
        R = self._autocorrelation_matrix(x_sinusoids, M)
        
        # Eigendecomposition
        eigenvals, eigenvecs = linalg.eigh(R)
        
        # Noise subspace (smallest eigenvalues)
        num_sinusoids = 2
        noise_subspace = eigenvecs[:, :M-num_sinusoids]
        
        # MUSIC spectrum
        freqs_music = np.linspace(0, 0.5, 1000)
        psd_music = np.zeros(len(freqs_music))
        
        for i, f in enumerate(freqs_music):
            a = np.exp(-1j * 2 * np.pi * f * np.arange(M))
            psd_music[i] = 1 / np.real(a.conj().T @ noise_subspace @ noise_subspace.conj().T @ a)
        
        results = {
            'ar_signal': x,
            'sinusoidal_signal': x_sinusoids,
            'periodogram': (freqs_p, psd_periodogram),
            'welch': (freqs_w, psd_welch),
            'multitaper': (freqs_mt, psd_multitaper),
            'ar_parametric': {
                'frequencies': freqs_ar,
                'psd': psd_ar,
                'coefficients': ar_coeffs_est,
                'noise_variance': noise_var_est
            },
            'music': {
                'frequencies': freqs_music,
                'psd': psd_music,
                'true_frequencies': [f1, f2]
            }
        }
        
        print(f"AR model order: {ar_order}")
        print(f"Estimated noise variance: {noise_var_est:.4f}")
        print(f"True noise variance: {noise_var:.4f}")
        print(f"MUSIC detected peaks near: {freqs_music[np.argsort(psd_music)[-2:]]}")
        
        return results
    
    def demonstrate_compressed_sensing(self):
        """
        Compressed sensing: Sparse signal recovery
        """
        print("\n=== COMPRESSED SENSING ===")
        
        # Sparse signal generation
        N = 256  # Signal dimension
        K = 10   # Sparsity level
        M = 64   # Number of measurements (M << N)
        
        # Create sparse signal
        x_true = np.zeros(N)
        support = np.random.choice(N, K, replace=False)
        x_true[support] = np.random.randn(K)
        
        # Measurement matrix (Gaussian)
        Phi = np.random.randn(M, N) / np.sqrt(M)
        
        # Measurements
        y = Phi @ x_true
        noise_level = 0.01
        y += noise_level * np.random.randn(M)
        
        # Recovery algorithms
        
        # 1. L1 minimization (Basis Pursuit)
        from scipy.optimize import minimize
        
        def l1_objective(x):
            return np.linalg.norm(x, 1)
        
        def l1_constraint(x):
            return np.linalg.norm(Phi @ x - y) - noise_level
        
        constraints = {'type': 'ineq', 'fun': l1_constraint}
        result_l1 = minimize(l1_objective, np.zeros(N), method='SLSQP', constraints=constraints)
        x_l1 = result_l1.x
        
        # 2. Orthogonal Matching Pursuit (OMP)
        omp = OrthogonalMatchingPursuit(n_nonzero_coefs=K)
        omp.fit(Phi, y)
        x_omp = omp.coef_
        
        # 3. Iterative Hard Thresholding
        x_iht = self._iterative_hard_thresholding(Phi, y, K, max_iter=100)
        
        # Performance metrics
        def recovery_error(x_est):
            return np.linalg.norm(x_est - x_true) / np.linalg.norm(x_true)
        
        error_l1 = recovery_error(x_l1)
        error_omp = recovery_error(x_omp)
        error_iht = recovery_error(x_iht)
        
        # Support recovery
        def support_recovery(x_est, threshold=1e-3):
            estimated_support = np.where(np.abs(x_est) > threshold)[0]
            true_support = np.where(np.abs(x_true) > 0)[0]
            
            intersection = len(np.intersect1d(estimated_support, true_support))
            precision = intersection / len(estimated_support) if len(estimated_support) > 0 else 0
            recall = intersection / len(true_support)
            
            return precision, recall
        
        prec_l1, rec_l1 = support_recovery(x_l1)
        prec_omp, rec_omp = support_recovery(x_omp)
        prec_iht, rec_iht = support_recovery(x_iht)
        
        # Restricted Isometry Property analysis
        delta = self._compute_rip_constant(Phi, 2*K)
        
        results = {
            'true_signal': x_true,
            'measurements': y,
            'measurement_matrix': Phi,
            'compression_ratio': M/N,
            'sparsity_ratio': K/N,
            'recovery_methods': {
                'l1_minimization': {
                    'signal': x_l1,
                    'error': error_l1,
                    'precision': prec_l1,
                    'recall': rec_l1
                },
                'omp': {
                    'signal': x_omp,
                    'error': error_omp,
                    'precision': prec_omp,
                    'recall': rec_omp
                },
                'iht': {
                    'signal': x_iht,
                    'error': error_iht,
                    'precision': prec_iht,
                    'recall': rec_iht
                }
            },
            'rip_constant': delta
        }
        
        print(f"Compression ratio (M/N): {M/N:.3f}")
        print(f"Sparsity ratio (K/N): {K/N:.3f}")
        print(f"L1 recovery error: {error_l1:.4f}")
        print(f"OMP recovery error: {error_omp:.4f}")
        print(f"IHT recovery error: {error_iht:.4f}")
        print(f"RIP constant δ_{2*K}: {delta:.4f}")
        
        return results
    
    def demonstrate_machine_learning_applications(self):
        """
        Signal processing applications in machine learning
        """
        print("\n=== MACHINE LEARNING APPLICATIONS ===")
        
        # 1. Feature extraction for time series
        N = 1000
        t = np.linspace(0, 10, N)
        
        # Multi-component signal
        signal_components = [
            np.sin(2*np.pi*2*t),  # Low frequency
            np.sin(2*np.pi*10*t) * np.exp(-t/5),  # Damped oscillation
            np.random.randn(N) * 0.2  # Noise
        ]
        x_ts = np.sum(signal_components, axis=0)
        
        # Feature extraction
        features = self._extract_signal_features(x_ts, fs=N/10)
        
        # 2. Blind Source Separation
        # Generate mixed signals
        n_sources = 3
        n_samples = 1000
        
        # Source signals
        s1 = np.sin(2*np.pi*0.1*np.arange(n_samples))  # Sinusoid
        s2 = signal.sawtooth(2*np.pi*0.05*np.arange(n_samples))  # Sawtooth
        s3 = np.random.randn(n_samples)  # Noise
        S = np.array([s1, s2, s3])
        
        # Mixing matrix
        A = np.random.randn(n_sources, n_sources)
        X = A @ S  # Mixed signals
        
        # ICA separation
        ica = FastICA(n_components=n_sources, random_state=42)
        S_estimated = ica.fit_transform(X.T).T
        W = ica.mixing_  # Estimated mixing matrix
        
        # Separation quality (correlation with original sources)
        correlations = np.abs([np.corrcoef(S[i], S_estimated[j])[0,1] 
                              for i in range(n_sources) 
                              for j in range(n_sources)])
        
        # 3. Convolutional features (1D convolution for signal processing)
        # Design filter bank
        fs = 100
        filter_bank = self._design_filter_bank(fs, n_filters=8)
        
        # Apply filter bank to signal
        filtered_signals = np.array([signal.lfilter(filt, 1, x_ts) for filt in filter_bank])
        
        # Feature vector: energy in each frequency band
        band_energies = np.array([np.mean(sig**2) for sig in filtered_signals])
        
        # 4. Spectral clustering using graph Laplacian
        # Create similarity matrix based on spectral coherence
        n_segments = 10
        segment_length = len(x_ts) // n_segments
        segments = [x_ts[i*segment_length:(i+1)*segment_length] 
                   for i in range(n_segments)]
        
        # Compute pairwise spectral similarities
        similarity_matrix = np.zeros((n_segments, n_segments))
        for i in range(n_segments):
            for j in range(n_segments):
                # Cross-spectral coherence
                f, Cxy = signal.coherence(segments[i], segments[j], fs=fs, nperseg=64)
                similarity_matrix[i,j] = np.mean(Cxy)
        
        # Graph Laplacian
        D = np.diag(np.sum(similarity_matrix, axis=1))
        L = D - similarity_matrix
        
        # Spectral clustering (using eigenvectors of Laplacian)
        eigenvals, eigenvecs = linalg.eigh(L)
        
        results = {
            'time_series': {
                'signal': x_ts,
                'components': signal_components,
                'features': features
            },
            'blind_separation': {
                'original_sources': S,
                'mixed_signals': X,
                'estimated_sources': S_estimated,
                'separation_quality': np.max(correlations.reshape(n_sources, n_sources), axis=1)
            },
            'filter_bank': {
                'filters': filter_bank,
                'filtered_signals': filtered_signals,
                'band_energies': band_energies
            },
            'spectral_clustering': {
                'similarity_matrix': similarity_matrix,
                'laplacian': L,
                'eigenvalues': eigenvals,
                'eigenvectors': eigenvecs
            }
        }
        
        print(f"Extracted {len(features)} time series features")
        print(f"BSS separation quality: {np.mean(np.max(correlations.reshape(n_sources, n_sources), axis=1)):.3f}")
        print(f"Filter bank: {len(filter_bank)} filters")
        print(f"Spectral clustering: {n_segments} segments analyzed")
        
        return results
    
    def demonstrate_advanced_topics(self):
        """
        Advanced signal processing topics: Graph signal processing, quantum signal processing
        """
        print("\n=== ADVANCED TOPICS ===")
        
        # Graph Signal Processing
        N = 50  # Number of nodes
        
        # Create random graph
        np.random.seed(42)
        adjacency = np.random.rand(N, N)
        adjacency = (adjacency + adjacency.T) / 2  # Symmetric
        adjacency[adjacency < 0.8] = 0  # Sparsify
        np.fill_diagonal(adjacency, 0)
        
        # Degree matrix and Laplacian
        degree = np.diag(np.sum(adjacency, axis=1))
        laplacian = degree - adjacency
        
        # Graph Fourier Transform (eigendecomposition of Laplacian)
        eigenvals, eigenvecs = linalg.eigh(laplacian)
        
        # Create graph signal (smooth on graph)
        x_graph = eigenvecs[:, :10] @ np.random.randn(10) * np.exp(-eigenvals[:10])
        
        # Graph Fourier Transform
        X_graph = eigenvecs.T @ x_graph
        
        # Graph filtering (low-pass)
        cutoff_freq = 5
        H_graph = np.exp(-eigenvals / cutoff_freq)
        x_filtered = eigenvecs @ (H_graph * X_graph)
        
        # Multiscale analysis on graphs
        scales = [0.5, 1.0, 2.0, 4.0]
        wavelet_coeffs = []
        for scale in scales:
            h_scale = np.exp(-eigenvals * scale) - np.exp(-eigenvals * scale * 2)
            coeff = eigenvecs @ (h_scale * X_graph)
            wavelet_coeffs.append(coeff)
        
        # Signal processing on manifolds (example: sphere)
        # Create points on unit sphere
        n_points = 100
        theta = np.random.uniform(0, 2*np.pi, n_points)
        phi = np.random.uniform(0, np.pi, n_points)
        
        # Spherical coordinates to Cartesian
        x_sphere = np.sin(phi) * np.cos(theta)
        y_sphere = np.sin(phi) * np.sin(theta)
        z_sphere = np.cos(phi)
        
        # Signal on sphere (using spherical harmonics basis)
        # Simplified: use first few spherical harmonics
        l_max = 3
        signal_sphere = np.zeros(n_points)
        for l in range(l_max + 1):
            for m in range(-l, l + 1):
                # Simplified spherical harmonic (real part)
                Y_lm = np.cos(m * theta) if m >= 0 else np.sin(abs(m) * theta)
                Y_lm *= np.sin(phi)**abs(m) * np.cos(phi)**(l-abs(m))
                signal_sphere += np.random.randn() * Y_lm
        
        # Diffusion on manifolds
        # Heat kernel approximation
        sigma = 0.1
        distances = np.sqrt((x_sphere[:, None] - x_sphere[None, :])**2 + 
                           (y_sphere[:, None] - y_sphere[None, :])**2 + 
                           (z_sphere[:, None] - z_sphere[None, :])**2)
        heat_kernel = np.exp(-distances**2 / (2 * sigma**2))
        
        # Diffusion filtering
        signal_diffused = heat_kernel @ signal_sphere / np.sum(heat_kernel, axis=1)
        
        results = {
            'graph_processing': {
                'adjacency': adjacency,
                'laplacian': laplacian,
                'eigenvalues': eigenvals,
                'eigenvectors': eigenvecs,
                'original_signal': x_graph,
                'filtered_signal': x_filtered,
                'wavelet_coeffs': wavelet_coeffs,
                'scales': scales
            },
            'manifold_processing': {
                'sphere_points': np.array([x_sphere, y_sphere, z_sphere]).T,
                'signal_original': signal_sphere,
                'signal_diffused': signal_diffused,
                'heat_kernel': heat_kernel
            }
        }
        
        print(f"Graph: {N} nodes, {np.sum(adjacency > 0)//2} edges")
        print(f"Laplacian eigenvalues range: [{eigenvals[1]:.3f}, {eigenvals[-1]:.3f}]")
        print(f"Graph signal smoothness: {x_graph.T @ laplacian @ x_graph:.3f}")
        print(f"Manifold signal processing on sphere with {n_points} points")
        
        return results
    
    # Helper methods
    def _compute_stft(self, x, window_size, hop_size, window=None):
        """Compute Short-Time Fourier Transform"""
        if window is None:
            window = np.hanning(window_size)
        
        n_frames = (len(x) - window_size) // hop_size + 1
        stft = np.zeros((n_frames, window_size//2 + 1), dtype=complex)
        
        for i in range(n_frames):
            start = i * hop_size
            frame = x[start:start + window_size] * window
            stft[i] = fft(frame)[:window_size//2 + 1]
        
        return stft
    
    def _dwt_decomposition(self, x, h_lpf, h_hpf, levels):
        """Discrete Wavelet Transform decomposition"""
        coeffs = []
        current = x.copy()
        
        for level in range(levels):
            # Convolution and downsampling
            low = signal.lfilter(h_lpf, 1, current)[::2]
            high = signal.lfilter(h_hpf, 1, current)[::2]
            
            coeffs.append(high)
            current = low
            
            if len(current) < 4:  # Stop if too short
                break
        
        coeffs.append(current)  # Approximation coefficients
        return coeffs[::-1]  # Reverse to get [approx, detail1, detail2, ...]
    
    def _dwt_reconstruction(self, coeffs, h_lpf, h_hpf):
        """Discrete Wavelet Transform reconstruction"""
        # Reconstruction filters
        g_lpf = h_lpf[::-1]
        g_hpf = h_hpf[::-1]
        
        # Start with approximation
        current = coeffs[0]
        
        # Iterate through detail coefficients
        for i in range(1, len(coeffs)):
            # Upsample
            current_up = np.zeros(2 * len(current))
            current_up[::2] = current
            
            detail_up = np.zeros(2 * len(coeffs[i]))
            detail_up[::2] = coeffs[i]
            
            # Filter and add
            current = (signal.lfilter(g_lpf, 1, current_up) + 
                      signal.lfilter(g_hpf, 1, detail_up))
        
        return current
    
    def _wavelet_denoise(self, coeffs, threshold):
        """Wavelet denoising using soft thresholding"""
        denoised = []
        
        # Keep approximation unchanged
        denoised.append(coeffs[0])
        
        # Apply soft thresholding to detail coefficients
        for i in range(1, len(coeffs)):
            detail = coeffs[i]
            denoised_detail = np.sign(detail) * np.maximum(np.abs(detail) - threshold, 0)
            denoised.append(denoised_detail)
        
        return denoised
    
    def _ar_yule_walker(self, x, order):
        """AR parameter estimation using Yule-Walker equations"""
        # Autocorrelation
        r = np.correlate(x, x, mode='full')
        r = r[len(r)//2:][:order+1]
        r = r / r[0]
        
        # Yule-Walker equations: R * a = r
        R = linalg.toeplitz(r[:-1])
        a = linalg.solve(R, r[1:])
        
        # Prediction error variance
        sigma2 = r[0] - np.dot(a, r[1:])
        
        return np.concatenate([[1], -a]), sigma2
    
    def _autocorrelation_matrix(self, x, M):
        """Compute autocorrelation matrix for MUSIC algorithm"""
        N = len(x)
        R = np.zeros((M, M), dtype=complex)
        
        for i in range(M):
            for j in range(M):
                for n in range(N - max(i, j)):
                    R[i, j] += x[n + i] * np.conj(x[n + j])
        
        return R / N
    
    def _iterative_hard_thresholding(self, Phi, y, K, max_iter=100, tol=1e-6):
        """Iterative Hard Thresholding for sparse recovery"""
        x = np.zeros(Phi.shape[1])
        
        for _ in range(max_iter):
            x_old = x.copy()
            
            # Gradient step
            gradient = Phi.T @ (Phi @ x - y)
            x = x - 0.1 * gradient
            
            # Hard thresholding
            indices = np.argsort(np.abs(x))[-K:]
            x_new = np.zeros_like(x)
            x_new[indices] = x[indices]
            x = x_new
            
            # Check convergence
            if np.linalg.norm(x - x_old) < tol:
                break
        
        return x
    
    def _compute_rip_constant(self, Phi, K, n_trials=1000):
        """Estimate Restricted Isometry Property constant"""
        deltas = []
        
        for _ in range(n_trials):
            # Random K-sparse vector
            support = np.random.choice(Phi.shape[1], K, replace=False)
            x = np.zeros(Phi.shape[1])
            x[support] = np.random.randn(K)
            
            # RIP condition
            norm_x = np.linalg.norm(x)
            norm_Phix = np.linalg.norm(Phi @ x)
            
            if norm_x > 0:
                ratio = norm_Phix / norm_x
                delta = max(abs(ratio - 1), abs(ratio**(-1) - 1))
                deltas.append(delta)
        
        return np.percentile(deltas, 95)  # 95th percentile estimate
    
    def _extract_signal_features(self, x, fs):
        """Extract comprehensive signal features"""
        features = {}
        
        # Time domain features
        features['mean'] = np.mean(x)
        features['std'] = np.std(x)
        features['skewness'] = self._skewness(x)
        features['kurtosis'] = self._kurtosis(x)
        features['energy'] = np.sum(x**2)
        features['rms'] = np.sqrt(np.mean(x**2))
        
        # Zero crossing rate
        features['zcr'] = np.sum(np.diff(np.sign(x)) != 0) / len(x)
        
        # Frequency domain features
        X = fft(x)
        freqs = fftfreq(len(x), 1/fs)
        psd = np.abs(X)**2
        
        # Spectral centroid
        features['spectral_centroid'] = np.sum(freqs[:len(freqs)//2] * psd[:len(psd)//2]) / np.sum(psd[:len(psd)//2])
        
        # Spectral bandwidth
        centroid = features['spectral_centroid']
        features['spectral_bandwidth'] = np.sqrt(np.sum((freqs[:len(freqs)//2] - centroid)**2 * 
                                                       psd[:len(psd)//2]) / np.sum(psd[:len(psd)//2]))
        
        # Spectral rolloff (95% of energy)
        cumsum_psd = np.cumsum(psd[:len(psd)//2])
        rolloff_idx = np.where(cumsum_psd >= 0.95 * cumsum_psd[-1])[0][0]
        features['spectral_rolloff'] = freqs[rolloff_idx]
        
        return features
    
    def _skewness(self, x):
        """Compute skewness"""
        mean_x = np.mean(x)
        std_x = np.std(x)
        return np.mean(((x - mean_x) / std_x)**3)
    
    def _kurtosis(self, x):
        """Compute kurtosis"""
        mean_x = np.mean(x)
        std_x = np.std(x)
        return np.mean(((x - mean_x) / std_x)**4) - 3
    
    def _design_filter_bank(self, fs, n_filters):
        """Design filter bank for feature extraction"""
        # Mel-scale frequency banks
        low_freq = 0
        high_freq = fs / 2
        
        # Mel scale conversion
        def hz_to_mel(hz):
            return 2595 * np.log10(1 + hz / 700)
        
        def mel_to_hz(mel):
            return 700 * (10**(mel / 2595) - 1)
        
        # Create mel-spaced frequency points
        mel_low = hz_to_mel(low_freq)
        mel_high = hz_to_mel(high_freq)
        mel_points = np.linspace(mel_low, mel_high, n_filters + 2)
        hz_points = mel_to_hz(mel_points)
        
        # Design triangular filters
        filters = []
        for i in range(n_filters):
            # Triangular filter
            left = hz_points[i]
            center = hz_points[i + 1]
            right = hz_points[i + 2]
            
            # Simple FIR filter approximation
            # (In practice, would use more sophisticated design)
            if center > 0:
                b = signal.firwin(64, [left, right], pass_zero=False, fs=fs)
                filters.append(b)
        
        return filters

# Demonstrate the comprehensive framework
framework = AdvancedSignalProcessingFramework()

print("ADVANCED SIGNAL PROCESSING FRAMEWORK")
print("="*50)

# Execute all demonstrations
signal_spaces = framework.demonstrate_signal_spaces_and_transforms()
fourier_analysis = framework.demonstrate_advanced_fourier_analysis()
wavelet_analysis = framework.demonstrate_wavelet_analysis()
filter_design = framework.demonstrate_digital_filter_design()
adaptive_filtering = framework.demonstrate_adaptive_filtering()
spectral_estimation = framework.demonstrate_spectral_estimation()
compressed_sensing = framework.demonstrate_compressed_sensing()
ml_applications = framework.demonstrate_machine_learning_applications()
advanced_topics = framework.demonstrate_advanced_topics()

print("\n" + "="*50)
print("SIGNAL PROCESSING FRAMEWORK DEMONSTRATION COMPLETE")
print("All advanced concepts implemented with mathematical rigor!")